In [3]:
import os
print(os.listdir('/workspace/data/raw'))

['.gitkeep', 'articles.csv', 'customers.csv', 'sample_submission.csv', 'transactions_train.csv']


In [4]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from math import log1p
from collections import defaultdict

DATA_DIR = Path('/workspace/data/raw')

# 線形結合の重み（デモと同じ）
SOURCE_WEIGHTS = {
    'history': 4.0,
    'category': 2.0,
    'global7': 2.0,
    'global14': 1.4,
    'global30': 0.9,
    'age': 0.8,
}
RANK_WEIGHT = 2.0
COUNT_WEIGHT = 0.25
RECENCY_WEIGHT = 3.0

CANDIDATE_LIMIT = 120
HISTORY_DAYS = 45
GLOBAL_POPULARITY_DAYS = [7, 14, 30]
AGE_POPULARITY_DAYS = 14
CATEGORY_POPULARITY_DAYS = 45
CATEGORY_HISTORY_DAYS = 120
CATEGORY_TOP_N = 16
CATEGORY_PREF_TOP_N = 2
CATEGORY_COLS = ['section_name', 'garment_group_name', 'index_group_name']

EVAL_CUTOFFS = [12, 24, 48, 96, 120]

print("設定完了")

設定完了


In [5]:
transactions = pl.read_csv(
    DATA_DIR / 'transactions_train.csv',
    columns=['t_dat', 'customer_id', 'article_id'],
    schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
).with_columns(pl.col('t_dat').str.to_date())

customers = pl.read_csv(
    DATA_DIR / 'customers.csv',
    columns=['customer_id', 'age'],
    schema_overrides={'customer_id': pl.String}
)

articles = pl.read_csv(
    DATA_DIR / 'articles.csv',
    columns=['article_id'] + CATEGORY_COLS,
    schema_overrides={'article_id': pl.String}
)

last_ts = transactions['t_dat'].max()
print(f"全データ最終日: {last_ts}")
print(f"transactions行数: {len(transactions):,}")

# 年齢バケット追加
customers = customers.with_columns(
    pl.when(pl.col('age').is_null()).then(pl.lit('unknown'))
    .when(pl.col('age') < 20).then(pl.lit('<20'))
    .when(pl.col('age') < 30).then(pl.lit('20s'))
    .when(pl.col('age') < 40).then(pl.lit('30s'))
    .when(pl.col('age') < 50).then(pl.lit('40s'))
    .when(pl.col('age') < 60).then(pl.lit('50s'))
    .otherwise(pl.lit('60+')).alias('age_bucket')
)

print("データ読み込み完了")

全データ最終日: 2020-09-22
transactions行数: 31,788,324
データ読み込み完了


In [6]:
def get_actuals(transactions, cutoff, label_days=7):
    """valid期間の購買を正解ラベルとして取得"""
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

def build_global_top(transactions, cutoff, days, limit):
    """全体人気商品リストを作成"""
    start = cutoff - timedelta(days=days)
    counts = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .group_by('article_id')
        .agg(pl.len().alias('cnt'))
        .sort('cnt', descending=True)
        .head(limit)
    )
    return [(row['article_id'], row['cnt']) for row in counts.iter_rows(named=True)]

def build_customer_history(transactions, cutoff, days, top_n):
    """ユーザーごとの購買履歴を取得"""
    start = cutoff - timedelta(days=days)
    recent = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .group_by(['customer_id', 'article_id'])
        .agg(pl.len().alias('cnt'), pl.max('t_dat').alias('last_date'))
        .sort(['customer_id', 'last_date', 'article_id'], descending=[False, True, False])
    )
    history = defaultdict(list)
    for row in recent.iter_rows(named=True):
        cid = row['customer_id']
        if len(history[cid]) < top_n:
            history[cid].append((row['article_id'], row['cnt'], (cutoff - row['last_date']).days))
    return dict(history)

def build_age_top(transactions, customers, cutoff, days, top_n):
    """年齢バケット別人気商品を作成"""
    start = cutoff - timedelta(days=days)
    recent = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .select(['customer_id', 'article_id'])
        .join(customers.select(['customer_id', 'age_bucket']), on='customer_id', how='left')
        .with_columns(pl.col('age_bucket').fill_null('unknown'))
    )
    counts = (
        recent.group_by(['age_bucket', 'article_id'])
        .agg(pl.len().alias('cnt'))
        .sort(['age_bucket', 'cnt'], descending=[False, True])
    )
    out = defaultdict(list)
    for row in counts.iter_rows(named=True):
        bucket = row['age_bucket']
        if len(out[bucket]) < top_n:
            out[bucket].append((row['article_id'], row['cnt']))
    return dict(out)

def build_category_top(transactions, articles, cutoff, days, top_n):
    """カテゴリ別人気商品を作成"""
    start = cutoff - timedelta(days=days)
    recent = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .select(['article_id'])
        .join(articles, on='article_id', how='left')
    )
    category_top = {}
    for col in CATEGORY_COLS:
        counts = (
            recent.filter(pl.col(col).is_not_null())
            .group_by([col, 'article_id'])
            .agg(pl.len().alias('cnt'))
            .sort([col, 'cnt'], descending=[False, True])
        )
        top_for_col = defaultdict(list)
        for row in counts.iter_rows(named=True):
            cat = row[col]
            if len(top_for_col[cat]) < top_n:
                top_for_col[cat].append((row['article_id'], row['cnt']))
        category_top[col] = dict(top_for_col)
    return category_top

def build_category_preferences(transactions, articles, cutoff, days, pref_top_n):
    """ユーザーごとの好みカテゴリを取得"""
    start = cutoff - timedelta(days=days)
    recent = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .select(['customer_id', 'article_id'])
        .join(articles, on='article_id', how='left')
    )
    preferences = {}
    for col in CATEGORY_COLS:
        counts = (
            recent.filter(pl.col(col).is_not_null())
            .group_by(['customer_id', col])
            .agg(pl.len().alias('cnt'))
            .sort(['customer_id', 'cnt'], descending=[False, True])
        )
        pref_for_col = defaultdict(list)
        for row in counts.iter_rows(named=True):
            cid = row['customer_id']
            if len(pref_for_col[cid]) < pref_top_n:
                pref_for_col[cid].append(row[col])
        preferences[col] = dict(pref_for_col)
    return preferences

def score_piece(source, rank, count, recency_days=None):
    """線形結合スコアを計算"""
    score = SOURCE_WEIGHTS[source] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
    if recency_days is not None:
        score += RECENCY_WEIGHT / (1.0 + max(recency_days, 0))
    return score

def add_candidate(candidates, article_id, source, rank, count, recency_days=None):
    """候補にスコアを追加"""
    if article_id not in candidates:
        if len(candidates) >= CANDIDATE_LIMIT:
            return
        candidates[article_id] = 0.0
    candidates[article_id] += score_piece(source, rank, count, recency_days)

def generate_candidates_linear(customer_id, components):
    """線形結合で候補を生成"""
    candidates = {}
    
    # history
    for rank, (article_id, count, recency_days) in enumerate(components['history'].get(customer_id, []), start=1):
        add_candidate(candidates, article_id, 'history', rank, count, recency_days)
    
    # age
    bucket = components['customer_age_bucket'].get(customer_id, 'unknown')
    for rank, (article_id, count) in enumerate(components['age_top'].get(bucket, []), start=1):
        add_candidate(candidates, article_id, 'age', rank, count)
    
    # global7
    for rank, (article_id, count) in enumerate(components['global_top'].get(7, []), start=1):
        add_candidate(candidates, article_id, 'global7', rank, count)
    
    # global14
    for rank, (article_id, count) in enumerate(components['global_top'].get(14, []), start=1):
        add_candidate(candidates, article_id, 'global14', rank, count)
    
    # global30
    for rank, (article_id, count) in enumerate(components['global_top'].get(30, []), start=1):
        add_candidate(candidates, article_id, 'global30', rank, count)
    
    # category
    for col in CATEGORY_COLS:
        for cat in components['category_preferences'][col].get(customer_id, []):
            for rank, (article_id, count) in enumerate(components['category_top'][col].get(cat, []), start=1):
                add_candidate(candidates, article_id, 'category', rank, count)
    
    # スコア降順でソート
    return sorted(candidates.items(), key=lambda x: -x[1])

def apk(actual_set, predicted, k):
    """AP@Kを計算"""
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in seen:
            continue
        seen.add(article_id)
        if article_id in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)

def evaluate(transactions, customers, articles, cutoff, label_days=7):
    """cutoffに対してRecall@KとMAP@Kを計算"""
    print(f"\ncutoff: {cutoff}")
    
    # 正解ラベル取得
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers = list(actuals.keys())
    print(f"valid_customers: {len(valid_customers):,}")
    
    # 各コンポーネント計算
    print("コンポーネント計算中...")
    components = {
        'history': build_customer_history(transactions, cutoff, HISTORY_DAYS, 24),
        'global_top': {days: build_global_top(transactions, cutoff, days, 50) for days in GLOBAL_POPULARITY_DAYS},
        'age_top': build_age_top(transactions, customers, cutoff, AGE_POPULARITY_DAYS, 40),
        'category_top': build_category_top(transactions, articles, cutoff, CATEGORY_POPULARITY_DAYS, CATEGORY_TOP_N),
        'category_preferences': build_category_preferences(transactions, articles, cutoff, CATEGORY_HISTORY_DAYS, CATEGORY_PREF_TOP_N),
        'customer_age_bucket': dict(zip(customers['customer_id'].to_list(), customers['age_bucket'].fill_null('unknown').to_list())),
    }
    
    # 評価
    print("評価中...")
    metric_acc = {k: {'recall': [], 'map': []} for k in EVAL_CUTOFFS}
    total_actual = sum(len(v) for v in actuals.values())
    
    for i, customer_id in enumerate(valid_customers):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers):,}")
        
        ordered = generate_candidates_linear(customer_id, components)
        prediction = [article_id for article_id, _ in ordered]
        actual_set = actuals[customer_id]
        
        for k in EVAL_CUTOFFS:
            hits = len(set(prediction[:k]) & actual_set)
            metric_acc[k]['recall'].append(hits)
            metric_acc[k]['map'].append(apk(actual_set, prediction, k))
    
    results = {}
    for k in EVAL_CUTOFFS:
        results[k] = {
            'recall': sum(metric_acc[k]['recall']) / total_actual,
            'map': sum(metric_acc[k]['map']) / len(valid_customers)
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("関数定義完了")

関数定義完了


In [7]:
from datetime import date, timedelta

# train cutoffs（3つ）
train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]

# valid cutoff（1つ）
valid_cutoff = date(2020, 9, 8)

# train評価（3cutoffの平均）
print("="*50)
print("TRAIN評価")
print("="*50)
train_results_all = []
for cutoff in train_cutoffs:
    result = evaluate(transactions, customers, articles, cutoff)
    train_results_all.append(result)

# train平均を計算
train_avg = {}
for k in EVAL_CUTOFFS:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

# valid評価
print("\n" + "="*50)
print("VALID評価")
print("="*50)
valid_results = evaluate(transactions, customers, articles, valid_cutoff)

# 結果を表形式で出力
print("\n" + "="*60)
print("結果サマリー")
print("="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

TRAIN評価

cutoff: 2020-08-18
valid_customers: 72,035
コンポーネント計算中...
評価中...
  進捗: 0 / 72,035
  進捗: 10,000 / 72,035
  進捗: 20,000 / 72,035
  進捗: 30,000 / 72,035
  進捗: 40,000 / 72,035
  進捗: 50,000 / 72,035
  進捗: 60,000 / 72,035
  進捗: 70,000 / 72,035
  Recall@12: 0.015423  MAP@12: 0.005385
  Recall@24: 0.026990  MAP@24: 0.006113
  Recall@48: 0.050008  MAP@48: 0.006945
  Recall@96: 0.094002  MAP@96: 0.007849
  Recall@120: 0.104486  MAP@120: 0.007980

cutoff: 2020-08-25
valid_customers: 80,253
コンポーネント計算中...
評価中...
  進捗: 0 / 80,253
  進捗: 10,000 / 80,253
  進捗: 20,000 / 80,253
  進捗: 30,000 / 80,253
  進捗: 40,000 / 80,253
  進捗: 50,000 / 80,253
  進捗: 60,000 / 80,253
  進捗: 70,000 / 80,253
  進捗: 80,000 / 80,253
  Recall@12: 0.019144  MAP@12: 0.007820
  Recall@24: 0.032500  MAP@24: 0.008746
  Recall@48: 0.059188  MAP@48: 0.009783
  Recall@96: 0.102139  MAP@96: 0.010627
  Recall@120: 0.112947  MAP@120: 0.010762

cutoff: 2020-09-01
valid_customers: 75,822
コンポーネント計算中...
評価中...
  進捗: 0 / 75,822
  進捗: 10,000

In [8]:
import implicit
from scipy.sparse import csr_matrix

# 直近データに絞る（計算効率のため）
CUTOFF = date(2020, 9, 8)  # validのcutoff
ALS_DAYS = 60  # 直近60日間のデータを使用

start_date = CUTOFF - timedelta(days=ALS_DAYS)
recent_tx = transactions.filter(
    (pl.col('t_dat') > pl.lit(start_date)) & 
    (pl.col('t_dat') <= pl.lit(CUTOFF))
)

# customer_id / article_idを整数インデックスに変換
customer_ids = recent_tx['customer_id'].unique().to_list()
article_ids = recent_tx['article_id'].unique().to_list()

customer_idx_map = {v: i for i, v in enumerate(customer_ids)}
article_idx_map = {v: i for i, v in enumerate(article_ids)}
article_idx_reverse = {i: v for v, i in article_idx_map.items()}

recent_tx_pd = recent_tx.to_pandas()
row = recent_tx_pd['customer_id'].map(customer_idx_map).values
col = recent_tx_pd['article_id'].map(article_idx_map).values
data = np.ones(len(recent_tx_pd))

# user×item疎行列
user_item_matrix = csr_matrix(
    (data, (row, col)),
    shape=(len(customer_ids), len(article_ids))
)

print(f"user×item行列サイズ: {user_item_matrix.shape}")
print(f"非ゼロ要素数: {user_item_matrix.nnz:,}")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


user×item行列サイズ: (400731, 36667)
非ゼロ要素数: 2,142,471


In [9]:
# Implicit ALSモデルの学習
model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=20,
    random_state=42
)

# item×user行列に転置（implicitの仕様）
item_user_matrix = user_item_matrix.T.tocsr()

print("ALS学習中...")
model.fit(item_user_matrix)
print("ALS学習完了")

/usr/local/lib/python3.11/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 24 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


ALS学習中...


100%|██████████| 20/20 [00:21<00:00,  1.06s/it]

ALS学習完了


In [10]:
# ユーザーごとにALS候補を生成
ALS_TOP_N = 50

print("ALS候補生成中...")
als_candidates = {}

# valid_customersのみに絞る
valid_customer_ids = list(actuals.keys()) if 'actuals' in dir() else customer_ids[:1000]

for i, customer_id in enumerate(valid_customer_ids):
    if i % 10000 == 0:
        print(f"  進捗: {i:,} / {len(valid_customer_ids):,}")
    
    if customer_id not in customer_idx_map:
        continue
    
    user_idx = customer_idx_map[customer_id]
    
    # ALS推薦
    recommended = model.recommend(
        user_idx,
        user_item_matrix[user_idx],
        N=ALS_TOP_N,
        filter_already_liked_items=False
    )
    
    article_indices = recommended[0]
    scores = recommended[1]
    
    als_candidates[customer_id] = [
        (article_idx_reverse[idx], float(score))
        for idx, score in zip(article_indices, scores)
        if idx in article_idx_reverse
    ]

print(f"ALS候補生成完了: {len(als_candidates):,}ユーザー")
print(f"サンプル（最初のユーザー）:")
first_user = list(als_candidates.keys())[0]
print(als_candidates[first_user][:5])

ALS候補生成中...
  進捗: 0 / 1,000
ALS候補生成完了: 1,000ユーザー
サンプル（最初のユーザー）:
[('0201219003', 0.0013049691915512085), ('0714493002', 0.0011682044714689255), ('0874837003', 0.0010684692533686757), ('0912312008', 0.00105677533429116), ('0895289001', 0.0010007848031818867)]


In [11]:
# ALS候補を線形結合に追加
ALS_WEIGHT = 1.5
ALS_TOP_N = 50

def generate_candidates_with_als(customer_id, components, als_candidates):
    """線形結合 + ALSで候補を生成"""
    candidates = {}
    
    # history
    for rank, (article_id, count, recency_days) in enumerate(components['history'].get(customer_id, []), start=1):
        add_candidate(candidates, article_id, 'history', rank, count, recency_days)
    
    # age
    bucket = components['customer_age_bucket'].get(customer_id, 'unknown')
    for rank, (article_id, count) in enumerate(components['age_top'].get(bucket, []), start=1):
        add_candidate(candidates, article_id, 'age', rank, count)
    
    # global7
    for rank, (article_id, count) in enumerate(components['global_top'].get(7, []), start=1):
        add_candidate(candidates, article_id, 'global7', rank, count)
    
    # global14
    for rank, (article_id, count) in enumerate(components['global_top'].get(14, []), start=1):
        add_candidate(candidates, article_id, 'global14', rank, count)
    
    # global30
    for rank, (article_id, count) in enumerate(components['global_top'].get(30, []), start=1):
        add_candidate(candidates, article_id, 'global30', rank, count)
    
    # category
    for col in CATEGORY_COLS:
        for cat in components['category_preferences'][col].get(customer_id, []):
            for rank, (article_id, count) in enumerate(components['category_top'][col].get(cat, []), start=1):
                add_candidate(candidates, article_id, 'category', rank, count)
    
    # ALS候補を追加
    for rank, (article_id, score) in enumerate(als_candidates.get(customer_id, []), start=1):
        if article_id not in candidates:
            if len(candidates) >= CANDIDATE_LIMIT:
                continue
            candidates[article_id] = 0.0
        candidates[article_id] += ALS_WEIGHT * score * 100  # スケール調整
    
    return sorted(candidates.items(), key=lambda x: -x[1])

def evaluate_with_als(transactions, customers, articles, cutoff, als_candidates, label_days=7):
    """ALS込みでRecall@KとMAP@Kを計算"""
    print(f"\ncutoff: {cutoff}")
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers = list(actuals.keys())
    print(f"valid_customers: {len(valid_customers):,}")
    
    print("コンポーネント計算中...")
    components = {
        'history': build_customer_history(transactions, cutoff, HISTORY_DAYS, 24),
        'global_top': {days: build_global_top(transactions, cutoff, days, 50) for days in GLOBAL_POPULARITY_DAYS},
        'age_top': build_age_top(transactions, customers, cutoff, AGE_POPULARITY_DAYS, 40),
        'category_top': build_category_top(transactions, articles, cutoff, CATEGORY_POPULARITY_DAYS, CATEGORY_TOP_N),
        'category_preferences': build_category_preferences(transactions, articles, cutoff, CATEGORY_HISTORY_DAYS, CATEGORY_PREF_TOP_N),
        'customer_age_bucket': dict(zip(customers['customer_id'].to_list(), customers['age_bucket'].fill_null('unknown').to_list())),
    }
    
    print("評価中...")
    metric_acc = {k: {'recall': [], 'map': []} for k in EVAL_CUTOFFS}
    total_actual = sum(len(v) for v in actuals.values())
    
    for i, customer_id in enumerate(valid_customers):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers):,}")
        
        ordered = generate_candidates_with_als(customer_id, components, als_candidates)
        prediction = [article_id for article_id, _ in ordered]
        actual_set = actuals[customer_id]
        
        for k in EVAL_CUTOFFS:
            hits = len(set(prediction[:k]) & actual_set)
            metric_acc[k]['recall'].append(hits)
            metric_acc[k]['map'].append(apk(actual_set, prediction, k))
    
    results = {}
    for k in EVAL_CUTOFFS:
        results[k] = {
            'recall': sum(metric_acc[k]['recall']) / total_actual,
            'map': sum(metric_acc[k]['map']) / len(valid_customers)
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("関数定義完了")

関数定義完了


In [12]:
from datetime import date, timedelta

# train cutoffs（3つ）
train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff = date(2020, 9, 8)

# 各cutoffでALSモデルを学習して評価
print("="*50)
print("TRAIN評価（ALS込み）")
print("="*50)

train_results_all = []

for cutoff in train_cutoffs:
    print(f"\n--- ALS学習 cutoff: {cutoff} ---")
    
    # cutoffに合わせたデータでALS学習
    start_date = cutoff - timedelta(days=60)
    recent_tx = transactions.filter(
        (pl.col('t_dat') > pl.lit(start_date)) &
        (pl.col('t_dat') <= pl.lit(cutoff))
    )
    
    c_ids = recent_tx['customer_id'].unique().to_list()
    a_ids = recent_tx['article_id'].unique().to_list()
    c_idx_map = {v: i for i, v in enumerate(c_ids)}
    a_idx_map = {v: i for i, v in enumerate(a_ids)}
    a_idx_rev = {i: v for v, i in a_idx_map.items()}
    
    recent_tx_pd = recent_tx.to_pandas()
    row = recent_tx_pd['customer_id'].map(c_idx_map).values
    col = recent_tx_pd['article_id'].map(a_idx_map).values
    data = np.ones(len(recent_tx_pd))
    
    uim = csr_matrix((data, (row, col)), shape=(len(c_ids), len(a_ids)))
    
    als_model = implicit.als.AlternatingLeastSquares(
        factors=64, regularization=0.01, iterations=20, random_state=42
    )
    als_model.fit(uim.T.tocsr())
    
    # valid_yユーザーのALS候補生成
    actuals_tmp = get_actuals(transactions, cutoff)
    valid_c_ids = list(actuals_tmp.keys())
    
    als_cands = {}
    for customer_id in valid_c_ids:
        if customer_id not in c_idx_map:
            continue
        user_idx = c_idx_map[customer_id]
        recommended = als_model.recommend(
            user_idx, uim[user_idx], N=ALS_TOP_N, filter_already_liked_items=False
        )
        als_cands[customer_id] = [
            (a_idx_rev[idx], float(score))
            for idx, score in zip(recommended[0], recommended[1])
            if idx in a_idx_rev
        ]
    
    result = evaluate_with_als(transactions, customers, articles, cutoff, als_cands)
    train_results_all.append(result)

# train平均
train_avg = {}
for k in EVAL_CUTOFFS:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

# valid評価
print("\n" + "="*50)
print("VALID評価（ALS込み）")
print("="*50)

# valid用ALS学習
start_date = valid_cutoff - timedelta(days=60)
recent_tx = transactions.filter(
    (pl.col('t_dat') > pl.lit(start_date)) &
    (pl.col('t_dat') <= pl.lit(valid_cutoff))
)

c_ids = recent_tx['customer_id'].unique().to_list()
a_ids = recent_tx['article_id'].unique().to_list()
c_idx_map = {v: i for i, v in enumerate(c_ids)}
a_idx_map = {v: i for i, v in enumerate(a_ids)}
a_idx_rev = {i: v for v, i in a_idx_map.items()}

recent_tx_pd = recent_tx.to_pandas()
row = recent_tx_pd['customer_id'].map(c_idx_map).values
col = recent_tx_pd['article_id'].map(a_idx_map).values
data = np.ones(len(recent_tx_pd))

uim = csr_matrix((data, (row, col)), shape=(len(c_ids), len(a_ids)))
als_model = implicit.als.AlternatingLeastSquares(
    factors=64, regularization=0.01, iterations=20, random_state=42
)
als_model.fit(uim.T.tocsr())

actuals_valid = get_actuals(transactions, valid_cutoff)
valid_c_ids = list(actuals_valid.keys())

als_cands_valid = {}
for customer_id in valid_c_ids:
    if customer_id not in c_idx_map:
        continue
    user_idx = c_idx_map[customer_id]
    recommended = als_model.recommend(
        user_idx, uim[user_idx], N=ALS_TOP_N, filter_already_liked_items=False
    )
    als_cands_valid[customer_id] = [
        (a_idx_rev[idx], float(score))
        for idx, score in zip(recommended[0], recommended[1])
        if idx in a_idx_rev
    ]

valid_results = evaluate_with_als(transactions, customers, articles, valid_cutoff, als_cands_valid)

# 結果表示
print("\n" + "="*60)
print("結果サマリー（ALS込み）")
print("="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

TRAIN評価（ALS込み）

--- ALS学習 cutoff: 2020-08-18 ---


100%|██████████| 20/20 [00:22<00:00,  1.14s/it]


IndexError: index 409360 is out of bounds for axis 0 with size 36889

In [ ]:
def build_als_candidates(transactions, cutoff, valid_customer_ids, als_top_n=50, als_days=60):
    """cutoffに対してALS候補を生成"""
    start_date = cutoff - timedelta(days=als_days)
    recent_tx = transactions.filter(
        (pl.col('t_dat') > pl.lit(start_date)) &
        (pl.col('t_dat') <= pl.lit(cutoff))
    )
    
    c_ids = recent_tx['customer_id'].unique().to_list()
    a_ids = recent_tx['article_id'].unique().to_list()
    c_idx_map = {v: i for i, v in enumerate(c_ids)}
    a_idx_map = {v: i for i, v in enumerate(a_ids)}
    a_idx_rev = {i: v for v, i in a_idx_map.items()}
    
    recent_tx_pd = recent_tx.to_pandas()
    row = recent_tx_pd['customer_id'].map(c_idx_map).values
    col = recent_tx_pd['article_id'].map(a_idx_map).values
    data = np.ones(len(recent_tx_pd))
    
    uim = csr_matrix((data, (row, col)), shape=(len(c_ids), len(a_ids)))
    
    als_model = implicit.als.AlternatingLeastSquares(
        factors=64, regularization=0.01, iterations=20, random_state=42
    )
    als_model.fit(uim.T.tocsr())
    
    n_users = uim.shape[0]
    als_cands = {}
    
    for customer_id in valid_customer_ids:
        if customer_id not in c_idx_map:
            continue
        user_idx = c_idx_map[customer_id]
        if user_idx >= n_users:
            continue
        
        recommended = als_model.recommend(
            user_idx, uim[user_idx], N=als_top_n, filter_already_liked_items=False
        )
        als_cands[customer_id] = [
            (a_idx_rev[idx], float(score))
            for idx, score in zip(recommended[0], recommended[1])
            if idx in a_idx_rev
        ]
    
    print(f"  ALS候補生成完了: {len(als_cands):,}ユーザー")
    return als_cands

print("関数定義完了")

関数定義完了


In [ ]:
from datetime import date, timedelta

train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff = date(2020, 9, 8)

# train評価
print("="*50)
print("TRAIN評価（ALS込み）")
print("="*50)

train_results_all = []

for cutoff in train_cutoffs:
    print(f"\n--- cutoff: {cutoff} ---")
    actuals_tmp = get_actuals(transactions, cutoff)
    valid_c_ids = list(actuals_tmp.keys())
    
    print(f"ALS学習中...")
    als_cands = build_als_candidates(transactions, cutoff, valid_c_ids)
    
    result = evaluate_with_als(transactions, customers, articles, cutoff, als_cands)
    train_results_all.append(result)

# train平均
train_avg = {}
for k in EVAL_CUTOFFS:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

# valid評価
print("\n" + "="*50)
print("VALID評価（ALS込み）")
print("="*50)

actuals_valid = get_actuals(transactions, valid_cutoff)
valid_c_ids = list(actuals_valid.keys())

print("ALS学習中...")
als_cands_valid = build_als_candidates(transactions, valid_cutoff, valid_c_ids)

valid_results = evaluate_with_als(transactions, customers, articles, valid_cutoff, als_cands_valid)

# 結果表示
print("\n" + "="*60)
print("結果サマリー（ALS込み）")
print("="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

TRAIN評価（ALS込み）

--- cutoff: 2020-08-18 ---
ALS学習中...


100%|██████████| 20/20 [00:22<00:00,  1.13s/it]


IndexError: index 124418 is out of bounds for axis 0 with size 36889

In [ ]:
def build_als_candidates(transactions, cutoff, valid_customer_ids, als_top_n=50, als_days=60):
    start_date = cutoff - timedelta(days=als_days)
    recent_tx = transactions.filter(
        (pl.col('t_dat') > pl.lit(start_date)) &
        (pl.col('t_dat') <= pl.lit(cutoff))
    )
    
    c_ids = recent_tx['customer_id'].unique().to_list()
    a_ids = recent_tx['article_id'].unique().to_list()
    c_idx_map = {v: i for i, v in enumerate(c_ids)}
    a_idx_map = {v: i for i, v in enumerate(a_ids)}
    a_idx_rev = {i: v for v, i in a_idx_map.items()}
    
    recent_tx_pd = recent_tx.to_pandas()
    row = recent_tx_pd['customer_id'].map(c_idx_map).values
    col = recent_tx_pd['article_id'].map(a_idx_map).values
    data = np.ones(len(recent_tx_pd))
    
    uim = csr_matrix((data, (row, col)), shape=(len(c_ids), len(a_ids)))
    
    als_model = implicit.als.AlternatingLeastSquares(
        factors=64, regularization=0.01, iterations=20, random_state=42
    )
    als_model.fit(uim.T.tocsr())
    
    # valid_customer_idsのうちc_idx_mapに存在するもののみ
    target_user_ids = [c for c in valid_customer_ids if c in c_idx_map]
    target_user_indices = np.array([c_idx_map[c] for c in target_user_ids])
    
    # バッチで一括推薦
    print(f"  ALS推薦中（{len(target_user_ids):,}ユーザー）...")
    recommended_ids, recommended_scores = als_model.recommend(
        target_user_indices,
        uim[target_user_indices],
        N=als_top_n,
        filter_already_liked_items=False
    )
    
    als_cands = {}
    for i, customer_id in enumerate(target_user_ids):
        als_cands[customer_id] = [
            (a_idx_rev[idx], float(score))
            for idx, score in zip(recommended_ids[i], recommended_scores[i])
            if idx in a_idx_rev
        ]
    
    print(f"  ALS候補生成完了: {len(als_cands):,}ユーザー")
    return als_cands

print("関数定義完了")

関数定義完了


In [ ]:
from datetime import date, timedelta

train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff = date(2020, 9, 8)

# train評価
print("="*50)
print("TRAIN評価（ALS込み）")
print("="*50)

train_results_all = []

for cutoff in train_cutoffs:
    print(f"\n--- cutoff: {cutoff} ---")
    actuals_tmp = get_actuals(transactions, cutoff)
    valid_c_ids = list(actuals_tmp.keys())
    
    als_cands = build_als_candidates(transactions, cutoff, valid_c_ids)
    result = evaluate_with_als(transactions, customers, articles, cutoff, als_cands)
    train_results_all.append(result)

# train平均
train_avg = {}
for k in EVAL_CUTOFFS:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

# valid評価
print("\n" + "="*50)
print("VALID評価（ALS込み）")
print("="*50)

actuals_valid = get_actuals(transactions, valid_cutoff)
valid_c_ids = list(actuals_valid.keys())
als_cands_valid = build_als_candidates(transactions, valid_cutoff, valid_c_ids)
valid_results = evaluate_with_als(transactions, customers, articles, valid_cutoff, als_cands_valid)

# 結果表示
print("\n" + "="*60)
print("結果サマリー（ALS込み）")
print("="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

TRAIN評価（ALS込み）

--- cutoff: 2020-08-18 ---


100%|██████████| 20/20 [00:22<00:00,  1.13s/it]

  ALS推薦中（48,440ユーザー）...


IndexError: index 292678 is out of bounds for axis 0 with size 36889

In [ ]:
# デバッグ用
start_date = date(2020, 8, 18) - timedelta(days=60)
recent_tx = transactions.filter(
    (pl.col('t_dat') > pl.lit(start_date)) &
    (pl.col('t_dat') <= pl.lit(date(2020, 8, 18)))
)

c_ids = recent_tx['customer_id'].unique().to_list()
a_ids = recent_tx['article_id'].unique().to_list()
c_idx_map = {v: i for i, v in enumerate(c_ids)}
a_idx_map = {v: i for i, v in enumerate(a_ids)}

recent_tx_pd = recent_tx.to_pandas()
row = recent_tx_pd['customer_id'].map(c_idx_map).values
col = recent_tx_pd['article_id'].map(a_idx_map).values
data = np.ones(len(recent_tx_pd))

uim = csr_matrix((data, (row, col)), shape=(len(c_ids), len(a_ids)))

als_model = implicit.als.AlternatingLeastSquares(
    factors=64, regularization=0.01, iterations=20, random_state=42
)
als_model.fit(uim.T.tocsr())

print(f"c_ids数: {len(c_ids):,}")
print(f"uim shape: {uim.shape}")
print(f"user_factors shape: {als_model.user_factors.shape}")
print(f"item_factors shape: {als_model.item_factors.shape}")

100%|██████████| 20/20 [00:22<00:00,  1.12s/it]

c_ids数: 431,363
uim shape: (431363, 36889)
user_factors shape: (36889, 64)
item_factors shape: (431363, 64)


In [ ]:
def build_als_candidates(transactions, cutoff, valid_customer_ids, als_top_n=50, als_days=60):
    start_date = cutoff - timedelta(days=als_days)
    recent_tx = transactions.filter(
        (pl.col('t_dat') > pl.lit(start_date)) &
        (pl.col('t_dat') <= pl.lit(cutoff))
    )
    
    c_ids = recent_tx['customer_id'].unique().to_list()
    a_ids = recent_tx['article_id'].unique().to_list()
    c_idx_map = {v: i for i, v in enumerate(c_ids)}
    a_idx_map = {v: i for i, v in enumerate(a_ids)}
    a_idx_rev = {i: v for v, i in a_idx_map.items()}
    
    recent_tx_pd = recent_tx.to_pandas()
    row = recent_tx_pd['customer_id'].map(c_idx_map).values
    col = recent_tx_pd['article_id'].map(a_idx_map).values
    data = np.ones(len(recent_tx_pd))
    
    # user × item行列
    uim = csr_matrix((data, (row, col)), shape=(len(c_ids), len(a_ids)))
    
    # implicitはitem × user行列を受け取るが
    # user_factorsはuser側、item_factorsはitem側になるように
    # uim（user×item）をそのままfitに渡す
    als_model = implicit.als.AlternatingLeastSquares(
        factors=64, regularization=0.01, iterations=20, random_state=42
    )
    als_model.fit(uim)  # ← .T を削除
    
    print(f"user_factors shape: {als_model.user_factors.shape}")
    print(f"item_factors shape: {als_model.item_factors.shape}")
    
    target_user_ids = [c for c in valid_customer_ids if c in c_idx_map]
    target_user_indices = np.array([c_idx_map[c] for c in target_user_ids])
    
    print(f"  ALS推薦中（{len(target_user_ids):,}ユーザー）...")
    recommended_ids, recommended_scores = als_model.recommend(
        target_user_indices,
        uim[target_user_indices],
        N=als_top_n,
        filter_already_liked_items=False
    )
    
    als_cands = {}
    for i, customer_id in enumerate(target_user_ids):
        als_cands[customer_id] = [
            (a_idx_rev[idx], float(score))
            for idx, score in zip(recommended_ids[i], recommended_scores[i])
            if idx in a_idx_rev
        ]
    
    print(f"  ALS候補生成完了: {len(als_cands):,}ユーザー")
    return als_cands

print("関数定義完了")

関数定義完了


In [ ]:
train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff = date(2020, 9, 8)

# train評価
print("="*50)
print("TRAIN評価（ALS込み）")
print("="*50)

train_results_all = []

for cutoff in train_cutoffs:
    print(f"\n--- cutoff: {cutoff} ---")
    actuals_tmp = get_actuals(transactions, cutoff)
    valid_c_ids = list(actuals_tmp.keys())
    als_cands = build_als_candidates(transactions, cutoff, valid_c_ids)
    result = evaluate_with_als(transactions, customers, articles, cutoff, als_cands)
    train_results_all.append(result)

# train平均
train_avg = {}
for k in EVAL_CUTOFFS:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

# valid評価
print("\n" + "="*50)
print("VALID評価（ALS込み）")
print("="*50)

actuals_valid = get_actuals(transactions, valid_cutoff)
valid_c_ids = list(actuals_valid.keys())
als_cands_valid = build_als_candidates(transactions, valid_cutoff, valid_c_ids)
valid_results = evaluate_with_als(transactions, customers, articles, valid_cutoff, als_cands_valid)

# 結果表示
print("\n" + "="*60)
print("結果サマリー（ALS込み）")
print("="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

TRAIN評価（ALS込み）

--- cutoff: 2020-08-18 ---


100%|██████████| 20/20 [00:22<00:00,  1.13s/it]


user_factors shape: (431363, 64)
item_factors shape: (36889, 64)
  ALS推薦中（48,440ユーザー）...
  ALS候補生成完了: 48,440ユーザー

cutoff: 2020-08-18
valid_customers: 72,035
コンポーネント計算中...
評価中...
  進捗: 0 / 72,035
  進捗: 10,000 / 72,035
  進捗: 20,000 / 72,035
  進捗: 30,000 / 72,035
  進捗: 40,000 / 72,035
  進捗: 50,000 / 72,035
  進捗: 60,000 / 72,035
  進捗: 70,000 / 72,035
  Recall@12: 0.018945  MAP@12: 0.007815
  Recall@24: 0.030261  MAP@24: 0.008524
  Recall@48: 0.052195  MAP@48: 0.009311
  Recall@96: 0.094305  MAP@96: 0.010175
  Recall@120: 0.105452  MAP@120: 0.010314

--- cutoff: 2020-08-25 ---


100%|██████████| 20/20 [00:21<00:00,  1.08s/it]


user_factors shape: (410700, 64)
item_factors shape: (36811, 64)
  ALS推薦中（51,446ユーザー）...
  ALS候補生成完了: 51,446ユーザー

cutoff: 2020-08-25
valid_customers: 80,253
コンポーネント計算中...
評価中...
  進捗: 0 / 80,253
  進捗: 10,000 / 80,253
  進捗: 20,000 / 80,253
  進捗: 30,000 / 80,253
  進捗: 40,000 / 80,253
  進捗: 50,000 / 80,253
  進捗: 60,000 / 80,253
  進捗: 70,000 / 80,253
  進捗: 80,000 / 80,253
  Recall@12: 0.021746  MAP@12: 0.009556
  Recall@24: 0.035114  MAP@24: 0.010465
  Recall@48: 0.060677  MAP@48: 0.011445
  Recall@96: 0.102108  MAP@96: 0.012266
  Recall@120: 0.113363  MAP@120: 0.012407

--- cutoff: 2020-09-01 ---


100%|██████████| 20/20 [00:21<00:00,  1.06s/it]


user_factors shape: (401635, 64)
item_factors shape: (36501, 64)
  ALS推薦中（47,792ユーザー）...
  ALS候補生成完了: 47,792ユーザー

cutoff: 2020-09-01
valid_customers: 75,822
コンポーネント計算中...
評価中...
  進捗: 0 / 75,822
  進捗: 10,000 / 75,822
  進捗: 20,000 / 75,822
  進捗: 30,000 / 75,822
  進捗: 40,000 / 75,822
  進捗: 50,000 / 75,822
  進捗: 60,000 / 75,822
  進捗: 70,000 / 75,822
  Recall@12: 0.026084  MAP@12: 0.010872
  Recall@24: 0.042357  MAP@24: 0.012006
  Recall@48: 0.067105  MAP@48: 0.012940
  Recall@96: 0.108008  MAP@96: 0.013794
  Recall@120: 0.119530  MAP@120: 0.013937

VALID評価（ALS込み）


100%|██████████| 20/20 [00:21<00:00,  1.06s/it]


user_factors shape: (400731, 64)
item_factors shape: (36667, 64)
  ALS推薦中（45,795ユーザー）...
  ALS候補生成完了: 45,795ユーザー

cutoff: 2020-09-08
valid_customers: 72,019
コンポーネント計算中...
評価中...
  進捗: 0 / 72,019
  進捗: 10,000 / 72,019
  進捗: 20,000 / 72,019
  進捗: 30,000 / 72,019
  進捗: 40,000 / 72,019
  進捗: 50,000 / 72,019
  進捗: 60,000 / 72,019
  進捗: 70,000 / 72,019
  Recall@12: 0.025400  MAP@12: 0.011891
  Recall@24: 0.039397  MAP@24: 0.012848
  Recall@48: 0.064517  MAP@48: 0.013770
  Recall@96: 0.108389  MAP@96: 0.014676
  Recall@120: 0.120372  MAP@120: 0.014824

結果サマリー（ALS込み）
split           k     recall        MAP
----------------------------------------
train avg      12   0.022259   0.009414
train avg      24   0.035910   0.010332
train avg      48   0.059992   0.011232
train avg      96   0.101474   0.012079
train avg     120   0.112782   0.012219
----------------------------------------
valid          12   0.025400   0.011891
valid          24   0.039397   0.012848
valid          48   0.064517   0

In [ ]:
def build_also_bought(transactions, cutoff, days=60, top_n=20):
    """商品ペアの共起頻度を計算"""
    start_date = cutoff - timedelta(days=days)
    recent = transactions.filter(
        (pl.col('t_dat') > pl.lit(start_date)) &
        (pl.col('t_dat') <= pl.lit(cutoff))
    ).select(['customer_id', 'article_id', 't_dat'])
    
    # 同じユーザー×同じ日に買われた商品ペアを作成
    print("  商品ペア計算中...")
    also_bought = (
        recent.join(recent, on=['customer_id', 't_dat'], how='inner')
        .filter(pl.col('article_id') != pl.col('article_id_right'))
        .group_by(['article_id', 'article_id_right'])
        .agg(pl.len().alias('count'))
        .sort(['article_id', 'count'], descending=[False, True])
    )
    
    # 商品ごとに上位N件を辞書に格納
    also_bought_dict = defaultdict(list)
    for row in also_bought.iter_rows(named=True):
        art = row['article_id']
        if len(also_bought_dict[art]) < top_n:
            also_bought_dict[art].append((row['article_id_right'], row['count']))
    
    print(f"  also bought計算完了: {len(also_bought_dict):,}商品")
    return dict(also_bought_dict)

print("関数定義完了")

関数定義完了


In [ ]:
ALSO_BOUGHT_WEIGHT = 1.5
ALSO_BOUGHT_DAYS = 60
ALSO_BOUGHT_TOP_N = 20

def generate_candidates_with_als_also_bought(customer_id, components, als_candidates):
    """線形結合 + ALS + also boughtで候補を生成"""
    candidates = {}
    
    # history
    for rank, (article_id, count, recency_days) in enumerate(components['history'].get(customer_id, []), start=1):
        add_candidate(candidates, article_id, 'history', rank, count, recency_days)
    
    # age
    bucket = components['customer_age_bucket'].get(customer_id, 'unknown')
    for rank, (article_id, count) in enumerate(components['age_top'].get(bucket, []), start=1):
        add_candidate(candidates, article_id, 'age', rank, count)
    
    # global7
    for rank, (article_id, count) in enumerate(components['global_top'].get(7, []), start=1):
        add_candidate(candidates, article_id, 'global7', rank, count)
    
    # global14
    for rank, (article_id, count) in enumerate(components['global_top'].get(14, []), start=1):
        add_candidate(candidates, article_id, 'global14', rank, count)
    
    # global30
    for rank, (article_id, count) in enumerate(components['global_top'].get(30, []), start=1):
        add_candidate(candidates, article_id, 'global30', rank, count)
    
    # category
    for col in CATEGORY_COLS:
        for cat in components['category_preferences'][col].get(customer_id, []):
            for rank, (article_id, count) in enumerate(components['category_top'][col].get(cat, []), start=1):
                add_candidate(candidates, article_id, 'category', rank, count)
    
    # ALS候補
    for rank, (article_id, score) in enumerate(als_candidates.get(customer_id, []), start=1):
        if article_id not in candidates:
            if len(candidates) >= CANDIDATE_LIMIT:
                continue
            candidates[article_id] = 0.0
        candidates[article_id] += ALS_WEIGHT * score * 100
    
    # also bought
    also_bought = components.get('also_bought', {})
    history_articles = [art for art, _, _ in components['history'].get(customer_id, [])]
    for hist_article in history_articles:
        for rank, (article_id, count) in enumerate(also_bought.get(hist_article, []), start=1):
            if article_id not in candidates:
                if len(candidates) >= CANDIDATE_LIMIT:
                    continue
                candidates[article_id] = 0.0
            candidates[article_id] += ALSO_BOUGHT_WEIGHT + RANK_WEIGHT / rank + COUNT_WEIGHT * log1p(count)
    
    return sorted(candidates.items(), key=lambda x: -x[1])


def evaluate_with_als_also_bought(transactions, customers, articles, cutoff, als_candidates, label_days=7):
    """ALS + also bought込みでRecall@KとMAP@Kを計算"""
    print(f"\ncutoff: {cutoff}")
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers = list(actuals.keys())
    print(f"valid_customers: {len(valid_customers):,}")
    
    print("コンポーネント計算中...")
    components = {
        'history': build_customer_history(transactions, cutoff, HISTORY_DAYS, 24),
        'global_top': {days: build_global_top(transactions, cutoff, days, 50) for days in GLOBAL_POPULARITY_DAYS},
        'age_top': build_age_top(transactions, customers, cutoff, AGE_POPULARITY_DAYS, 40),
        'category_top': build_category_top(transactions, articles, cutoff, CATEGORY_POPULARITY_DAYS, CATEGORY_TOP_N),
        'category_preferences': build_category_preferences(transactions, articles, cutoff, CATEGORY_HISTORY_DAYS, CATEGORY_PREF_TOP_N),
        'customer_age_bucket': dict(zip(customers['customer_id'].to_list(), customers['age_bucket'].fill_null('unknown').to_list())),
        'also_bought': build_also_bought(transactions, cutoff, ALSO_BOUGHT_DAYS, ALSO_BOUGHT_TOP_N),
    }
    
    print("評価中...")
    metric_acc = {k: {'recall': [], 'map': []} for k in EVAL_CUTOFFS}
    total_actual = sum(len(v) for v in actuals.values())
    
    for i, customer_id in enumerate(valid_customers):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers):,}")
        
        ordered = generate_candidates_with_als_also_bought(customer_id, components, als_candidates)
        prediction = [article_id for article_id, _ in ordered]
        actual_set = actuals[customer_id]
        
        for k in EVAL_CUTOFFS:
            hits = len(set(prediction[:k]) & actual_set)
            metric_acc[k]['recall'].append(hits)
            metric_acc[k]['map'].append(apk(actual_set, prediction, k))
    
    results = {}
    for k in EVAL_CUTOFFS:
        results[k] = {
            'recall': sum(metric_acc[k]['recall']) / total_actual,
            'map': sum(metric_acc[k]['map']) / len(valid_customers)
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("関数定義完了")

関数定義完了


In [ ]:
train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff = date(2020, 9, 8)

# train評価
print("="*50)
print("TRAIN評価（ALS + also bought込み）")
print("="*50)

train_results_all = []

for cutoff in train_cutoffs:
    print(f"\n--- cutoff: {cutoff} ---")
    actuals_tmp = get_actuals(transactions, cutoff)
    valid_c_ids = list(actuals_tmp.keys())
    als_cands = build_als_candidates(transactions, cutoff, valid_c_ids)
    result = evaluate_with_als_also_bought(transactions, customers, articles, cutoff, als_cands)
    train_results_all.append(result)

# train平均
train_avg = {}
for k in EVAL_CUTOFFS:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

# valid評価
print("\n" + "="*50)
print("VALID評価（ALS + also bought込み）")
print("="*50)

actuals_valid = get_actuals(transactions, valid_cutoff)
valid_c_ids = list(actuals_valid.keys())
als_cands_valid = build_als_candidates(transactions, valid_cutoff, valid_c_ids)
valid_results = evaluate_with_als_also_bought(transactions, customers, articles, valid_cutoff, als_cands_valid)

# 結果表示
print("\n" + "="*60)
print("結果サマリー（ALS + also bought込み）")
print("="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

TRAIN評価（ALS + also bought込み）

--- cutoff: 2020-08-18 ---


100%|██████████| 20/20 [00:22<00:00,  1.12s/it]


user_factors shape: (431363, 64)
item_factors shape: (36889, 64)
  ALS推薦中（48,440ユーザー）...
  ALS候補生成完了: 48,440ユーザー

cutoff: 2020-08-18
valid_customers: 72,035
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 36,069商品
評価中...
  進捗: 0 / 72,035
  進捗: 10,000 / 72,035
  進捗: 20,000 / 72,035
  進捗: 30,000 / 72,035
  進捗: 40,000 / 72,035
  進捗: 50,000 / 72,035
  進捗: 60,000 / 72,035
  進捗: 70,000 / 72,035
  Recall@12: 0.020180  MAP@12: 0.008338
  Recall@24: 0.032302  MAP@24: 0.009108
  Recall@48: 0.056255  MAP@48: 0.009989
  Recall@96: 0.094574  MAP@96: 0.010776
  Recall@120: 0.105296  MAP@120: 0.010910

--- cutoff: 2020-08-25 ---


100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


user_factors shape: (410700, 64)
item_factors shape: (36811, 64)
  ALS推薦中（51,446ユーザー）...
  ALS候補生成完了: 51,446ユーザー

cutoff: 2020-08-25
valid_customers: 80,253
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 35,946商品
評価中...
  進捗: 0 / 80,253
  進捗: 10,000 / 80,253
  進捗: 20,000 / 80,253
  進捗: 30,000 / 80,253
  進捗: 40,000 / 80,253
  進捗: 50,000 / 80,253
  進捗: 60,000 / 80,253
  進捗: 70,000 / 80,253
  進捗: 80,000 / 80,253
  Recall@12: 0.022926  MAP@12: 0.009802
  Recall@24: 0.037053  MAP@24: 0.010777
  Recall@48: 0.063761  MAP@48: 0.011825
  Recall@96: 0.101539  MAP@96: 0.012564
  Recall@120: 0.112015  MAP@120: 0.012696

--- cutoff: 2020-09-01 ---


100%|██████████| 20/20 [00:21<00:00,  1.06s/it]


user_factors shape: (401635, 64)
item_factors shape: (36501, 64)
  ALS推薦中（47,792ユーザー）...
  ALS候補生成完了: 47,792ユーザー

cutoff: 2020-09-01
valid_customers: 75,822
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 35,607商品
評価中...
  進捗: 0 / 75,822
  進捗: 10,000 / 75,822
  進捗: 20,000 / 75,822
  進捗: 30,000 / 75,822
  進捗: 40,000 / 75,822
  進捗: 50,000 / 75,822
  進捗: 60,000 / 75,822
  進捗: 70,000 / 75,822
  Recall@12: 0.026941  MAP@12: 0.011247
  Recall@24: 0.044205  MAP@24: 0.012472
  Recall@48: 0.071003  MAP@48: 0.013508
  Recall@96: 0.108668  MAP@96: 0.014290
  Recall@120: 0.119946  MAP@120: 0.014429

VALID評価（ALS + also bought込み）


100%|██████████| 20/20 [00:21<00:00,  1.06s/it]


user_factors shape: (400731, 64)
item_factors shape: (36667, 64)
  ALS推薦中（45,795ユーザー）...
  ALS候補生成完了: 45,795ユーザー

cutoff: 2020-09-08
valid_customers: 72,019
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 35,766商品
評価中...
  進捗: 0 / 72,019
  進捗: 10,000 / 72,019
  進捗: 20,000 / 72,019
  進捗: 30,000 / 72,019
  進捗: 40,000 / 72,019
  進捗: 50,000 / 72,019
  進捗: 60,000 / 72,019
  進捗: 70,000 / 72,019
  Recall@12: 0.026370  MAP@12: 0.012219
  Recall@24: 0.041569  MAP@24: 0.013299
  Recall@48: 0.068646  MAP@48: 0.014320
  Recall@96: 0.109113  MAP@96: 0.015149
  Recall@120: 0.120578  MAP@120: 0.015288

結果サマリー（ALS + also bought込み）
split           k     recall        MAP
----------------------------------------
train avg      12   0.023349   0.009796
train avg      24   0.037853   0.010785
train avg      48   0.063673   0.011774
train avg      96   0.101594   0.012543
train avg     120   0.112419   0.012678
----------------------------------------
valid          12   0.026370   0.012219
valid          24

In [ ]:
from itertools import product

# 探索する重みの候補
param_grid = {
    'history': [3.0, 4.0, 5.0],
    'category': [1.0, 2.0, 3.0],
    'global7': [1.0, 2.0, 3.0],
    'age': [0.5, 1.0, 1.5],
    'als': [1.0, 1.5, 2.0],
    'also_bought': [1.0, 1.5, 2.0],
}

# valid cutoffで事前にコンポーネントを計算（毎回計算しないように）
print("コンポーネント事前計算中...")
cutoff = date(2020, 9, 8)
actuals_gs = get_actuals(transactions, cutoff)
valid_c_ids_gs = list(actuals_gs.keys())
total_actual_gs = sum(len(v) for v in actuals_gs.values())

als_cands_gs = build_als_candidates(transactions, cutoff, valid_c_ids_gs)

components_gs = {
    'history': build_customer_history(transactions, cutoff, HISTORY_DAYS, 24),
    'global_top': {days: build_global_top(transactions, cutoff, days, 50) for days in GLOBAL_POPULARITY_DAYS},
    'age_top': build_age_top(transactions, customers, cutoff, AGE_POPULARITY_DAYS, 40),
    'category_top': build_category_top(transactions, articles, cutoff, CATEGORY_POPULARITY_DAYS, CATEGORY_TOP_N),
    'category_preferences': build_category_preferences(transactions, articles, cutoff, CATEGORY_HISTORY_DAYS, CATEGORY_PREF_TOP_N),
    'customer_age_bucket': dict(zip(customers['customer_id'].to_list(), customers['age_bucket'].fill_null('unknown').to_list())),
    'also_bought': build_also_bought(transactions, cutoff, ALSO_BOUGHT_DAYS, ALSO_BOUGHT_TOP_N),
}

print("事前計算完了！")
print(f"グリッドサーチの組み合わせ数: {3**6:,}")

コンポーネント事前計算中...


100%|██████████| 20/20 [00:20<00:00,  1.04s/it]


user_factors shape: (400731, 64)
item_factors shape: (36667, 64)
  ALS推薦中（45,795ユーザー）...
  ALS候補生成完了: 45,795ユーザー
  商品ペア計算中...
  also bought計算完了: 35,766商品
事前計算完了！
グリッドサーチの組み合わせ数: 729


In [ ]:
def evaluate_with_weights(components, als_candidates, actuals, total_actual, weights):
    """指定した重みでMAP@12を計算"""
    map_acc = []
    
    for customer_id in actuals.keys():
        candidates = {}
        
        # history
        for rank, (article_id, count, recency_days) in enumerate(components['history'].get(customer_id, []), start=1):
            score = weights['history'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
            score += RECENCY_WEIGHT / (1.0 + max(recency_days, 0))
            if article_id not in candidates:
                if len(candidates) >= CANDIDATE_LIMIT:
                    continue
                candidates[article_id] = 0.0
            candidates[article_id] += score
        
        # age
        bucket = components['customer_age_bucket'].get(customer_id, 'unknown')
        for rank, (article_id, count) in enumerate(components['age_top'].get(bucket, []), start=1):
            score = weights['age'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
            if article_id not in candidates:
                if len(candidates) >= CANDIDATE_LIMIT:
                    continue
                candidates[article_id] = 0.0
            candidates[article_id] += score
        
        # global7
        for rank, (article_id, count) in enumerate(components['global_top'].get(7, []), start=1):
            score = weights['global7'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
            if article_id not in candidates:
                if len(candidates) >= CANDIDATE_LIMIT:
                    continue
                candidates[article_id] = 0.0
            candidates[article_id] += score
        
        # global14
        for rank, (article_id, count) in enumerate(components['global_top'].get(14, []), start=1):
            score = weights['global14'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
            if article_id not in candidates:
                if len(candidates) >= CANDIDATE_LIMIT:
                    continue
                candidates[article_id] = 0.0
            candidates[article_id] += score
        
        # global30
        for rank, (article_id, count) in enumerate(components['global_top'].get(30, []), start=1):
            score = weights['global30'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
            if article_id not in candidates:
                if len(candidates) >= CANDIDATE_LIMIT:
                    continue
                candidates[article_id] = 0.0
            candidates[article_id] += score
        
        # category
        for col in CATEGORY_COLS:
            for cat in components['category_preferences'][col].get(customer_id, []):
                for rank, (article_id, count) in enumerate(components['category_top'][col].get(cat, []), start=1):
                    score = weights['category'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
                    if article_id not in candidates:
                        if len(candidates) >= CANDIDATE_LIMIT:
                            continue
                        candidates[article_id] = 0.0
                    candidates[article_id] += score
        
        # ALS
        for rank, (article_id, score) in enumerate(als_candidates.get(customer_id, []), start=1):
            als_score = weights['als'] * score * 100
            if article_id not in candidates:
                if len(candidates) >= CANDIDATE_LIMIT:
                    continue
                candidates[article_id] = 0.0
            candidates[article_id] += als_score
        
        # also bought
        history_articles = [art for art, _, _ in components['history'].get(customer_id, [])]
        for hist_article in history_articles:
            for rank, (article_id, count) in enumerate(components['also_bought'].get(hist_article, []), start=1):
                score = weights['also_bought'] + RANK_WEIGHT / rank + COUNT_WEIGHT * log1p(count)
                if article_id not in candidates:
                    if len(candidates) >= CANDIDATE_LIMIT:
                        continue
                    candidates[article_id] = 0.0
                candidates[article_id] += score
        
        ordered = sorted(candidates.items(), key=lambda x: -x[1])
        prediction = [article_id for article_id, _ in ordered]
        actual_set = actuals[customer_id]
        map_acc.append(apk(actual_set, prediction, 12))
    
    return np.mean(map_acc)

# グリッドサーチ実行
print("グリッドサーチ開始...")
best_map = 0
best_weights = None
results_gs = []

keys = list(param_grid.keys())
values = list(param_grid.values())
total_combinations = 1
for v in values:
    total_combinations *= len(v)

print(f"総組み合わせ数: {total_combinations:,}")

for i, combo in enumerate(product(*values)):
    weights = dict(zip(keys, combo))
    # global14, global30は固定
    weights['global14'] = 1.4
    weights['global30'] = 0.9
    
    map12 = evaluate_with_weights(components_gs, als_cands_gs, actuals_gs, total_actual_gs, weights)
    results_gs.append((map12, weights.copy()))
    
    if map12 > best_map:
        best_map = map12
        best_weights = weights.copy()
        print(f"  [{i+1}/{total_combinations}] 新しいベスト MAP@12: {best_map:.6f} weights: {weights}")

print(f"\nグリッドサーチ完了!")
print(f"最良MAP@12: {best_map:.6f}")
print(f"最良重み: {best_weights}")

グリッドサーチ開始...
総組み合わせ数: 729
  [1/729] 新しいベスト MAP@12: 0.011995 weights: {'history': 3.0, 'category': 1.0, 'global7': 1.0, 'age': 0.5, 'als': 1.0, 'also_bought': 1.0, 'global14': 1.4, 'global30': 0.9}
  [2/729] 新しいベスト MAP@12: 0.012078 weights: {'history': 3.0, 'category': 1.0, 'global7': 1.0, 'age': 0.5, 'als': 1.0, 'also_bought': 1.5, 'global14': 1.4, 'global30': 0.9}
  [3/729] 新しいベスト MAP@12: 0.012158 weights: {'history': 3.0, 'category': 1.0, 'global7': 1.0, 'age': 0.5, 'als': 1.0, 'also_bought': 2.0, 'global14': 1.4, 'global30': 0.9}
  [4/729] 新しいベスト MAP@12: 0.012253 weights: {'history': 3.0, 'category': 1.0, 'global7': 1.0, 'age': 0.5, 'als': 1.5, 'also_bought': 1.0, 'global14': 1.4, 'global30': 0.9}
  [5/729] 新しいベスト MAP@12: 0.012305 weights: {'history': 3.0, 'category': 1.0, 'global7': 1.0, 'age': 0.5, 'als': 1.5, 'also_bought': 1.5, 'global14': 1.4, 'global30': 0.9}
  [6/729] 新しいベスト MAP@12: 0.012388 weights: {'history': 3.0, 'category': 1.0, 'global7': 1.0, 'age': 0.5, 'als': 1.5, 'a

In [ ]:
# 最良重みをSOURCE_WEIGHTSに反映
SOURCE_WEIGHTS['history'] = best_weights['history']
SOURCE_WEIGHTS['category'] = best_weights['category']
SOURCE_WEIGHTS['global7'] = best_weights['global7']
SOURCE_WEIGHTS['age'] = best_weights['age']
ALS_WEIGHT = best_weights['als']
ALSO_BOUGHT_WEIGHT = best_weights['also_bought']

print(f"最良重み適用完了: {best_weights}")

# train/valid全評価
train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff = date(2020, 9, 8)

print("\n" + "="*50)
print("TRAIN評価（最良重み）")
print("="*50)

train_results_all = []
for cutoff in train_cutoffs:
    print(f"\n--- cutoff: {cutoff} ---")
    actuals_tmp = get_actuals(transactions, cutoff)
    valid_c_ids = list(actuals_tmp.keys())
    als_cands = build_als_candidates(transactions, cutoff, valid_c_ids)
    result = evaluate_with_als_also_bought(transactions, customers, articles, cutoff, als_cands)
    train_results_all.append(result)

train_avg = {}
for k in EVAL_CUTOFFS:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

print("\n" + "="*50)
print("VALID評価（最良重み）")
print("="*50)

actuals_valid = get_actuals(transactions, valid_cutoff)
valid_c_ids = list(actuals_valid.keys())
als_cands_valid = build_als_candidates(transactions, valid_cutoff, valid_c_ids)
valid_results = evaluate_with_als_also_bought(transactions, customers, articles, valid_cutoff, als_cands_valid)

print("\n" + "="*60)
print("結果サマリー（最良重み）")
print("="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

最良重み適用完了: {'history': 5.0, 'category': 1.0, 'global7': 1.0, 'age': 0.5, 'als': 2.0, 'also_bought': 2.0, 'global14': 1.4, 'global30': 0.9}

TRAIN評価（最良重み）

--- cutoff: 2020-08-18 ---


100%|██████████| 20/20 [00:22<00:00,  1.12s/it]


user_factors shape: (431363, 64)
item_factors shape: (36889, 64)
  ALS推薦中（48,440ユーザー）...
  ALS候補生成完了: 48,440ユーザー

cutoff: 2020-08-18
valid_customers: 72,035
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 36,069商品
評価中...
  進捗: 0 / 72,035
  進捗: 10,000 / 72,035
  進捗: 20,000 / 72,035
  進捗: 30,000 / 72,035
  進捗: 40,000 / 72,035
  進捗: 50,000 / 72,035
  進捗: 60,000 / 72,035
  進捗: 70,000 / 72,035
  Recall@12: 0.021640  MAP@12: 0.009015
  Recall@24: 0.034390  MAP@24: 0.009849
  Recall@48: 0.059521  MAP@48: 0.010794
  Recall@96: 0.093850  MAP@96: 0.011487
  Recall@120: 0.105374  MAP@120: 0.011629

--- cutoff: 2020-08-25 ---


100%|██████████| 20/20 [00:21<00:00,  1.08s/it]


user_factors shape: (410700, 64)
item_factors shape: (36811, 64)
  ALS推薦中（51,446ユーザー）...
  ALS候補生成完了: 51,446ユーザー

cutoff: 2020-08-25
valid_customers: 80,253
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 35,946商品
評価中...
  進捗: 0 / 80,253
  進捗: 10,000 / 80,253
  進捗: 20,000 / 80,253
  進捗: 30,000 / 80,253
  進捗: 40,000 / 80,253
  進捗: 50,000 / 80,253
  進捗: 60,000 / 80,253
  進捗: 70,000 / 80,253
  進捗: 80,000 / 80,253
  Recall@12: 0.024031  MAP@12: 0.010279
  Recall@24: 0.038915  MAP@24: 0.011326
  Recall@48: 0.066806  MAP@48: 0.012446
  Recall@96: 0.098463  MAP@96: 0.013059
  Recall@120: 0.113261  MAP@120: 0.013240

--- cutoff: 2020-09-01 ---


100%|██████████| 20/20 [00:21<00:00,  1.06s/it]


user_factors shape: (401635, 64)
item_factors shape: (36501, 64)
  ALS推薦中（47,792ユーザー）...
  ALS候補生成完了: 47,792ユーザー

cutoff: 2020-09-01
valid_customers: 75,822
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 35,607商品
評価中...
  進捗: 0 / 75,822
  進捗: 10,000 / 75,822
  進捗: 20,000 / 75,822
  進捗: 30,000 / 75,822
  進捗: 40,000 / 75,822
  進捗: 50,000 / 75,822
  進捗: 60,000 / 75,822
  進捗: 70,000 / 75,822
  Recall@12: 0.028168  MAP@12: 0.011691
  Recall@24: 0.045792  MAP@24: 0.012968
  Recall@48: 0.074179  MAP@48: 0.014119
  Recall@96: 0.108181  MAP@96: 0.014810
  Recall@120: 0.120106  MAP@120: 0.014954

VALID評価（最良重み）


100%|██████████| 20/20 [00:21<00:00,  1.05s/it]


user_factors shape: (400731, 64)
item_factors shape: (36667, 64)
  ALS推薦中（45,795ユーザー）...
  ALS候補生成完了: 45,795ユーザー

cutoff: 2020-09-08
valid_customers: 72,019
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 35,766商品
評価中...
  進捗: 0 / 72,019
  進捗: 10,000 / 72,019
  進捗: 20,000 / 72,019
  進捗: 30,000 / 72,019
  進捗: 40,000 / 72,019
  進捗: 50,000 / 72,019
  進捗: 60,000 / 72,019
  進捗: 70,000 / 72,019
  Recall@12: 0.027520  MAP@12: 0.012827
  Recall@24: 0.043776  MAP@24: 0.014024
  Recall@48: 0.072932  MAP@48: 0.015167
  Recall@96: 0.107832  MAP@96: 0.015857
  Recall@120: 0.120504  MAP@120: 0.016016

結果サマリー（最良重み）
split           k     recall        MAP
----------------------------------------
train avg      12   0.024613   0.010328
train avg      24   0.039699   0.011381
train avg      48   0.066835   0.012453
train avg      96   0.100165   0.013119
train avg     120   0.112914   0.013274
----------------------------------------
valid          12   0.027520   0.012827
valid          24   0.043776   0

In [ ]:
def build_item2item_candidates(transactions, cutoff, customer_history, als_days=60, top_n=20):
    """item2item ALSで候補を生成"""
    start_date = cutoff - timedelta(days=als_days)
    recent_tx = transactions.filter(
        (pl.col('t_dat') > pl.lit(start_date)) &
        (pl.col('t_dat') <= pl.lit(cutoff))
    )
    
    c_ids = recent_tx['customer_id'].unique().to_list()
    a_ids = recent_tx['article_id'].unique().to_list()
    c_idx_map = {v: i for i, v in enumerate(c_ids)}
    a_idx_map = {v: i for i, v in enumerate(a_ids)}
    a_idx_rev = {i: v for v, i in a_idx_map.items()}
    
    recent_tx_pd = recent_tx.to_pandas()
    row = recent_tx_pd['customer_id'].map(c_idx_map).values
    col = recent_tx_pd['article_id'].map(a_idx_map).values
    data = np.ones(len(recent_tx_pd))
    
    uim = csr_matrix((data, (row, col)), shape=(len(c_ids), len(a_ids)))
    
    # item2item: item×userで学習
    als_model = implicit.als.AlternatingLeastSquares(
        factors=64, regularization=0.01, iterations=20, random_state=42
    )
    als_model.fit(uim.T.tocsr())  # item×userで学習
    
    print(f"  item_factors shape: {als_model.item_factors.shape}")
    
    # ユーザーの購買履歴の商品に類似した商品を取得
    item2item_cands = {}
    
    for customer_id, hist in customer_history.items():
        if not hist:
            continue
        
        candidates = {}
        for article_id, count, _ in hist:
            if article_id not in a_idx_map:
                continue
            
            item_idx = a_idx_map[article_id]
            
            # 類似商品を取得
            similar_ids, similar_scores = als_model.similar_items(
                item_idx, N=top_n + 1
            )
            
            for idx, score in zip(similar_ids, similar_scores):
                if idx == item_idx:
                    continue
                if idx not in a_idx_rev:
                    continue
                sim_article = a_idx_rev[idx]
                if sim_article not in candidates:
                    candidates[sim_article] = 0.0
                candidates[sim_article] += float(score)
        
        if candidates:
            item2item_cands[customer_id] = sorted(
                candidates.items(), key=lambda x: -x[1]
            )[:top_n]
    
    print(f"  item2item候補生成完了: {len(item2item_cands):,}ユーザー")
    return item2item_cands

print("関数定義完了")

関数定義完了


In [ ]:
ITEM2ITEM_WEIGHT = 1.5

def generate_candidates_full(customer_id, components, als_candidates, item2item_candidates):
    """全戦略を組み合わせた候補生成"""
    candidates = {}
    
    # history
    for rank, (article_id, count, recency_days) in enumerate(components['history'].get(customer_id, []), start=1):
        score = best_weights['history'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
        score += RECENCY_WEIGHT / (1.0 + max(recency_days, 0))
        if article_id not in candidates:
            if len(candidates) >= CANDIDATE_LIMIT:
                continue
            candidates[article_id] = 0.0
        candidates[article_id] += score
    
    # age
    bucket = components['customer_age_bucket'].get(customer_id, 'unknown')
    for rank, (article_id, count) in enumerate(components['age_top'].get(bucket, []), start=1):
        score = best_weights['age'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
        if article_id not in candidates:
            if len(candidates) >= CANDIDATE_LIMIT:
                continue
            candidates[article_id] = 0.0
        candidates[article_id] += score
    
    # global7
    for rank, (article_id, count) in enumerate(components['global_top'].get(7, []), start=1):
        score = best_weights['global7'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
        if article_id not in candidates:
            if len(candidates) >= CANDIDATE_LIMIT:
                continue
            candidates[article_id] = 0.0
        candidates[article_id] += score
    
    # global14
    for rank, (article_id, count) in enumerate(components['global_top'].get(14, []), start=1):
        score = best_weights['global14'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
        if article_id not in candidates:
            if len(candidates) >= CANDIDATE_LIMIT:
                continue
            candidates[article_id] = 0.0
        candidates[article_id] += score
    
    # global30
    for rank, (article_id, count) in enumerate(components['global_top'].get(30, []), start=1):
        score = best_weights['global30'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
        if article_id not in candidates:
            if len(candidates) >= CANDIDATE_LIMIT:
                continue
            candidates[article_id] = 0.0
        candidates[article_id] += score
    
    # category
    for col in CATEGORY_COLS:
        for cat in components['category_preferences'][col].get(customer_id, []):
            for rank, (article_id, count) in enumerate(components['category_top'][col].get(cat, []), start=1):
                score = best_weights['category'] + RANK_WEIGHT / max(rank, 1) + COUNT_WEIGHT * log1p(max(count, 0))
                if article_id not in candidates:
                    if len(candidates) >= CANDIDATE_LIMIT:
                        continue
                    candidates[article_id] = 0.0
                candidates[article_id] += score
    
    # user2item ALS
    for rank, (article_id, score) in enumerate(als_candidates.get(customer_id, []), start=1):
        als_score = best_weights['als'] * score * 100
        if article_id not in candidates:
            if len(candidates) >= CANDIDATE_LIMIT:
                continue
            candidates[article_id] = 0.0
        candidates[article_id] += als_score
    
    # also bought
    history_articles = [art for art, _, _ in components['history'].get(customer_id, [])]
    for hist_article in history_articles:
        for rank, (article_id, count) in enumerate(components['also_bought'].get(hist_article, []), start=1):
            score = best_weights['also_bought'] + RANK_WEIGHT / rank + COUNT_WEIGHT * log1p(count)
            if article_id not in candidates:
                if len(candidates) >= CANDIDATE_LIMIT:
                    continue
                candidates[article_id] = 0.0
            candidates[article_id] += score
    
    # item2item ALS
    for rank, (article_id, score) in enumerate(item2item_candidates.get(customer_id, []), start=1):
        i2i_score = ITEM2ITEM_WEIGHT * score
        if article_id not in candidates:
            if len(candidates) >= CANDIDATE_LIMIT:
                continue
            candidates[article_id] = 0.0
        candidates[article_id] += i2i_score
    
    return sorted(candidates.items(), key=lambda x: -x[1])


def evaluate_full(transactions, customers, articles, cutoff, als_candidates, label_days=7):
    """全戦略込みでRecall@KとMAP@Kを計算"""
    print(f"\ncutoff: {cutoff}")
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers = list(actuals.keys())
    print(f"valid_customers: {len(valid_customers):,}")
    
    print("コンポーネント計算中...")
    components = {
        'history': build_customer_history(transactions, cutoff, HISTORY_DAYS, 24),
        'global_top': {days: build_global_top(transactions, cutoff, days, 50) for days in GLOBAL_POPULARITY_DAYS},
        'age_top': build_age_top(transactions, customers, cutoff, AGE_POPULARITY_DAYS, 40),
        'category_top': build_category_top(transactions, articles, cutoff, CATEGORY_POPULARITY_DAYS, CATEGORY_TOP_N),
        'category_preferences': build_category_preferences(transactions, articles, cutoff, CATEGORY_HISTORY_DAYS, CATEGORY_PREF_TOP_N),
        'customer_age_bucket': dict(zip(customers['customer_id'].to_list(), customers['age_bucket'].fill_null('unknown').to_list())),
        'also_bought': build_also_bought(transactions, cutoff, ALSO_BOUGHT_DAYS, ALSO_BOUGHT_TOP_N),
    }
    
    # item2item候補生成
    print("item2item ALS計算中...")
    item2item_cands = build_item2item_candidates(
        transactions, cutoff, components['history']
    )
    
    print("評価中...")
    metric_acc = {k: {'recall': [], 'map': []} for k in EVAL_CUTOFFS}
    total_actual = sum(len(v) for v in actuals.values())
    
    for i, customer_id in enumerate(valid_customers):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers):,}")
        
        ordered = generate_candidates_full(customer_id, components, als_candidates, item2item_cands)
        prediction = [article_id for article_id, _ in ordered]
        actual_set = actuals[customer_id]
        
        for k in EVAL_CUTOFFS:
            hits = len(set(prediction[:k]) & actual_set)
            metric_acc[k]['recall'].append(hits)
            metric_acc[k]['map'].append(apk(actual_set, prediction, k))
    
    results = {}
    for k in EVAL_CUTOFFS:
        results[k] = {
            'recall': sum(metric_acc[k]['recall']) / total_actual,
            'map': sum(metric_acc[k]['map']) / len(valid_customers)
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("関数定義完了")

関数定義完了


In [ ]:
train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff = date(2020, 9, 8)

# train評価
print("="*50)
print("TRAIN評価（全戦略）")
print("="*50)

train_results_all = []

for cutoff in train_cutoffs:
    print(f"\n--- cutoff: {cutoff} ---")
    actuals_tmp = get_actuals(transactions, cutoff)
    valid_c_ids = list(actuals_tmp.keys())
    als_cands = build_als_candidates(transactions, cutoff, valid_c_ids)
    result = evaluate_full(transactions, customers, articles, cutoff, als_cands)
    train_results_all.append(result)

# train平均
train_avg = {}
for k in EVAL_CUTOFFS:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

# valid評価
print("\n" + "="*50)
print("VALID評価（全戦略）")
print("="*50)

actuals_valid = get_actuals(transactions, valid_cutoff)
valid_c_ids = list(actuals_valid.keys())
als_cands_valid = build_als_candidates(transactions, valid_cutoff, valid_c_ids)
valid_results = evaluate_full(transactions, customers, articles, valid_cutoff, als_cands_valid)

# 結果表示
print("\n" + "="*60)
print("結果サマリー（全戦略）")
print("="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

TRAIN評価（全戦略）

--- cutoff: 2020-08-18 ---


100%|██████████| 20/20 [00:22<00:00,  1.13s/it]


user_factors shape: (431363, 64)
item_factors shape: (36889, 64)
  ALS推薦中（48,440ユーザー）...
  ALS候補生成完了: 48,440ユーザー

cutoff: 2020-08-18
valid_customers: 72,035
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 36,069商品
item2item ALS計算中...


100%|██████████| 20/20 [00:22<00:00,  1.13s/it]


  item_factors shape: (431363, 64)


KeyboardInterrupt: 

In [ ]:
def evaluate_rulebase_only(transactions, customers, articles, cutoff, label_days=7):
    """ルールベースのみでRecall@KとMAP@Kを計算"""
    print(f"\ncutoff: {cutoff}")
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers = list(actuals.keys())
    print(f"valid_customers: {len(valid_customers):,}")
    
    print("コンポーネント計算中...")
    components = {
        'history': build_customer_history(transactions, cutoff, HISTORY_DAYS, 24),
        'global_top': {days: build_global_top(transactions, cutoff, days, 50) for days in GLOBAL_POPULARITY_DAYS},
        'age_top': build_age_top(transactions, customers, cutoff, AGE_POPULARITY_DAYS, 40),
        'category_top': build_category_top(transactions, articles, cutoff, CATEGORY_POPULARITY_DAYS, CATEGORY_TOP_N),
        'category_preferences': build_category_preferences(transactions, articles, cutoff, CATEGORY_HISTORY_DAYS, CATEGORY_PREF_TOP_N),
        'customer_age_bucket': dict(zip(customers['customer_id'].to_list(), customers['age_bucket'].fill_null('unknown').to_list())),
        'also_bought': build_also_bought(transactions, cutoff, ALSO_BOUGHT_DAYS, ALSO_BOUGHT_TOP_N),
    }
    
    print("評価中...")
    metric_acc = {k: {'recall': [], 'map': []} for k in EVAL_CUTOFFS}
    total_actual = sum(len(v) for v in actuals.values())
    
    for i, customer_id in enumerate(valid_customers):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers):,}")
        
        ordered = generate_candidates_with_als_also_bought(customer_id, components, {})
        prediction = [article_id for article_id, _ in ordered]
        actual_set = actuals[customer_id]
        
        for k in EVAL_CUTOFFS:
            hits = len(set(prediction[:k]) & actual_set)
            metric_acc[k]['recall'].append(hits)
            metric_acc[k]['map'].append(apk(actual_set, prediction, k))
    
    results = {}
    for k in EVAL_CUTOFFS:
        results[k] = {
            'recall': sum(metric_acc[k]['recall']) / total_actual,
            'map': sum(metric_acc[k]['map']) / len(valid_customers)
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

# 評価実行
train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff = date(2020, 9, 8)

print("="*50)
print("TRAIN評価（ルールベースのみ）")
print("="*50)

train_results_all = []
for cutoff in train_cutoffs:
    print(f"\n--- cutoff: {cutoff} ---")
    result = evaluate_rulebase_only(transactions, customers, articles, cutoff)
    train_results_all.append(result)

train_avg = {}
for k in EVAL_CUTOFFS:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

print("\n" + "="*50)
print("VALID評価（ルールベースのみ）")
print("="*50)
valid_results = evaluate_rulebase_only(transactions, customers, articles, valid_cutoff)

# 結果表示
print("\n" + "="*60)
print("結果サマリー（ルールベースのみ）")
print("="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

TRAIN評価（ルールベースのみ）

--- cutoff: 2020-08-18 ---

cutoff: 2020-08-18
valid_customers: 72,035
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 36,069商品
評価中...
  進捗: 0 / 72,035
  進捗: 10,000 / 72,035
  進捗: 20,000 / 72,035
  進捗: 30,000 / 72,035
  進捗: 40,000 / 72,035
  進捗: 50,000 / 72,035
  進捗: 60,000 / 72,035
  進捗: 70,000 / 72,035
  Recall@12: 0.019495  MAP@12: 0.007273
  Recall@24: 0.033441  MAP@24: 0.008234
  Recall@48: 0.060145  MAP@48: 0.009253
  Recall@96: 0.094106  MAP@96: 0.009933
  Recall@120: 0.105448  MAP@120: 0.010074

--- cutoff: 2020-08-25 ---

cutoff: 2020-08-25
valid_customers: 80,253
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 35,946商品
評価中...
  進捗: 0 / 80,253
  進捗: 10,000 / 80,253
  進捗: 20,000 / 80,253
  進捗: 30,000 / 80,253
  進捗: 40,000 / 80,253
  進捗: 50,000 / 80,253
  進捗: 60,000 / 80,253
  進捗: 70,000 / 80,253
  進捗: 80,000 / 80,253
  Recall@12: 0.022796  MAP@12: 0.009016
  Recall@24: 0.038852  MAP@24: 0.010183
  Recall@48: 0.067272  MAP@48: 0.011341
  Recall@96: 0.098584  MAP@9

In [ ]:
# 候補数の分布を確認
test_actuals = get_actuals(transactions, valid_cutoff)
test_customer = list(test_actuals.keys())[0]
ordered = generate_candidates_with_als_also_bought(test_customer, components_gs, {})
print(f"候補数: {len(ordered)}")
print(f"上位12件: {[a for a, _ in ordered[:12]]}")

候補数: 120
上位12件: ['0850917001', '0706016001', '0751471001', '0915526001', '0751471043', '0762846031', '0918292001', '0448509014', '0762846006', '0915529003', '0896152002', '0863595006']


In [ ]:
def evaluate_rulebase_v2(transactions, customers, articles, cutoff, label_days=7):
    """デモと同じRecall(customer mean)とMAP@Kを計算"""
    print(f"\ncutoff: {cutoff}")
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"valid_customers: {len(valid_customers_list):,}")
    
    print("コンポーネント計算中...")
    components = {
        'history': build_customer_history(transactions, cutoff, HISTORY_DAYS, 24),
        'global_top': {days: build_global_top(transactions, cutoff, days, 50) for days in GLOBAL_POPULARITY_DAYS},
        'age_top': build_age_top(transactions, customers, cutoff, AGE_POPULARITY_DAYS, 40),
        'category_top': build_category_top(transactions, articles, cutoff, CATEGORY_POPULARITY_DAYS, CATEGORY_TOP_N),
        'category_preferences': build_category_preferences(transactions, articles, cutoff, CATEGORY_HISTORY_DAYS, CATEGORY_PREF_TOP_N),
        'customer_age_bucket': dict(zip(customers['customer_id'].to_list(), customers['age_bucket'].fill_null('unknown').to_list())),
        'also_bought': build_also_bought(transactions, cutoff, ALSO_BOUGHT_DAYS, ALSO_BOUGHT_TOP_N),
    }
    
    print("評価中...")
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in EVAL_CUTOFFS}
    
    for i, customer_id in enumerate(valid_customers_list):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers_list):,}")
        
        ordered = generate_candidates_with_als_also_bought(customer_id, components, {})
        prediction = [article_id for article_id, _ in ordered]
        actual_set = actuals[customer_id]
        
        for k in EVAL_CUTOFFS:
            hits = len(set(prediction[:k]) & actual_set)
            # デモと同じ：ユーザーごとのrecallを計算
            metric_acc[k]['customer_recall'].append(hits / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, prediction, k))
    
    results = {}
    for k in EVAL_CUTOFFS:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

# 評価実行
train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff = date(2020, 9, 8)

print("="*50)
print("TRAIN評価")
print("="*50)

train_results_all = []
for cutoff in train_cutoffs:
    print(f"\n--- cutoff: {cutoff} ---")
    result = evaluate_rulebase_v2(transactions, customers, articles, cutoff)
    train_results_all.append(result)

train_avg = {}
for k in EVAL_CUTOFFS:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

print("\n" + "="*50)
print("VALID評価")
print("="*50)
valid_results = evaluate_rulebase_v2(transactions, customers, articles, valid_cutoff)

# 結果表示
print("\n" + "="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in EVAL_CUTOFFS:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

TRAIN評価

--- cutoff: 2020-08-18 ---

cutoff: 2020-08-18
valid_customers: 72,035
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 36,069商品
評価中...
  進捗: 0 / 72,035
  進捗: 10,000 / 72,035
  進捗: 20,000 / 72,035
  進捗: 30,000 / 72,035
  進捗: 40,000 / 72,035
  進捗: 50,000 / 72,035
  進捗: 60,000 / 72,035
  進捗: 70,000 / 72,035
  Recall@12: 0.021010  MAP@12: 0.007198
  Recall@24: 0.036534  MAP@24: 0.008153
  Recall@48: 0.067663  MAP@48: 0.009172
  Recall@96: 0.105602  MAP@96: 0.009856
  Recall@120: 0.117799  MAP@120: 0.009997

--- cutoff: 2020-08-25 ---

cutoff: 2020-08-25
valid_customers: 80,253
コンポーネント計算中...
  商品ペア計算中...
  also bought計算完了: 35,946商品
評価中...
  進捗: 0 / 80,253
  進捗: 10,000 / 80,253
  進捗: 20,000 / 80,253
  進捗: 30,000 / 80,253
  進捗: 40,000 / 80,253
  進捗: 50,000 / 80,253
  進捗: 60,000 / 80,253
  進捗: 70,000 / 80,253
  進捗: 80,000 / 80,253
  Recall@12: 0.026451  MAP@12: 0.009023
  Recall@24: 0.045682  MAP@24: 0.010190
  Recall@48: 0.080854  MAP@48: 0.011360
  Recall@96: 0.113819  MAP@96: 0.01195

In [ ]:
def evaluate_single_strategy(strategy_name, get_candidates_fn, actuals, k=12):
    """単一戦略のRecall@kを計算"""
    hits = []
    for customer_id, actual_set in actuals.items():
        candidates = get_candidates_fn(customer_id)
        hit = len(set(candidates[:k]) & actual_set)
        hits.append(hit / len(actual_set) if actual_set else 0.0)
    recall = np.mean(hits)
    print(f"  {strategy_name:<30} Recall@{k}: {recall:.6f}")
    return recall

# validのコンポーネントを事前計算
print("コンポーネント計算中...")
cutoff = date(2020, 9, 8)
actuals_eval = get_actuals(transactions, cutoff)

components_eval = {
    'history': build_customer_history(transactions, cutoff, HISTORY_DAYS, 24),
    'global_top': {days: build_global_top(transactions, cutoff, days, 50) for days in GLOBAL_POPULARITY_DAYS},
    'age_top': build_age_top(transactions, customers, cutoff, AGE_POPULARITY_DAYS, 40),
    'category_top': build_category_top(transactions, articles, cutoff, CATEGORY_POPULARITY_DAYS, CATEGORY_TOP_N),
    'category_preferences': build_category_preferences(transactions, articles, cutoff, CATEGORY_HISTORY_DAYS, CATEGORY_PREF_TOP_N),
    'customer_age_bucket': dict(zip(customers['customer_id'].to_list(), customers['age_bucket'].fill_null('unknown').to_list())),
    'also_bought': build_also_bought(transactions, cutoff, ALSO_BOUGHT_DAYS, ALSO_BOUGHT_TOP_N),
}

print("各戦略のRecall@12計算中...")
print("-"*50)

results_single = {}

# ① history
results_single['history'] = evaluate_single_strategy(
    'history',
    lambda cid: [art for art, _, _ in components_eval['history'].get(cid, [])],
    actuals_eval
)

# ② global7
results_single['global7'] = evaluate_single_strategy(
    'global7',
    lambda cid: [art for art, _ in components_eval['global_top'].get(7, [])],
    actuals_eval
)

# ③ global14
results_single['global14'] = evaluate_single_strategy(
    'global14',
    lambda cid: [art for art, _ in components_eval['global_top'].get(14, [])],
    actuals_eval
)

# ④ global30
results_single['global30'] = evaluate_single_strategy(
    'global30',
    lambda cid: [art for art, _ in components_eval['global_top'].get(30, [])],
    actuals_eval
)

# ⑤ age
results_single['age'] = evaluate_single_strategy(
    'age',
    lambda cid: [art for art, _ in components_eval['age_top'].get(
        components_eval['customer_age_bucket'].get(cid, 'unknown'), []
    )],
    actuals_eval
)

# ⑥ category
results_single['category'] = evaluate_single_strategy(
    'category',
    lambda cid: list(dict.fromkeys([
        art
        for col in CATEGORY_COLS
        for cat in components_eval['category_preferences'][col].get(cid, [])
        for art, _ in components_eval['category_top'][col].get(cat, [])
    ])),
    actuals_eval
)

# ⑦ also_bought
results_single['also_bought'] = evaluate_single_strategy(
    'also_bought',
    lambda cid: list(dict.fromkeys([
        art
        for hist_art, _, _ in components_eval['history'].get(cid, [])
        for art, _ in components_eval['also_bought'].get(hist_art, [])
    ])),
    actuals_eval
)

# 結果を降順でソート
print("\n" + "="*50)
print("戦略別Recall@12ランキング")
print("="*50)
for name, recall in sorted(results_single.items(), key=lambda x: -x[1]):
    print(f"  {name:<30} {recall:.6f}")

コンポーネント計算中...


NameError: name 'date' is not defined

In [ ]:
def evaluate_history_with_fallback(actuals, components_eval, k=12):
    """historyで候補生成、足りない分をglobal7で補完"""
    hits = []
    
    global7_list = [art for art, _ in components_eval['global_top'].get(7, [])]
    
    for customer_id, actual_set in actuals.items():
        # history候補
        history_candidates = [art for art, _, _ in components_eval['history'].get(customer_id, [])]
        
        # 足りない分をglobal7で補完
        remaining = k - len(history_candidates)
        if remaining > 0:
            history_set = set(history_candidates)
            fill = [art for art in global7_list if art not in history_set][:remaining]
            candidates = history_candidates + fill
        else:
            candidates = history_candidates[:k]
        
        hit = len(set(candidates[:k]) & actual_set)
        hits.append(hit / len(actual_set) if actual_set else 0.0)
    
    recall = np.mean(hits)
    print(f"  history + global7補完  Recall@{k}: {recall:.6f}")
    return recall

result = evaluate_history_with_fallback(actuals_eval, components_eval)

  history + global7補完  Recall@12: 0.049095


In [ ]:
def evaluate_history_days(transactions, customers, articles, cutoff, days_list, k=12):
    """historyの期間を変えてRecall@12を計算"""
    actuals_tmp = get_actuals(transactions, cutoff)
    global7_list = [art for art, _ in build_global_top(transactions, cutoff, 7, 50)]
    
    print(f"cutoff: {cutoff}")
    print("-"*50)
    
    for days in days_list:
        history = build_customer_history(transactions, cutoff, days, 24)
        
        hits = []
        for customer_id, actual_set in actuals_tmp.items():
            history_candidates = [art for art, _, _ in history.get(customer_id, [])]
            remaining = k - len(history_candidates)
            if remaining > 0:
                history_set = set(history_candidates)
                fill = [art for art in global7_list if art not in history_set][:remaining]
                candidates = history_candidates + fill
            else:
                candidates = history_candidates[:k]
            
            hit = len(set(candidates[:k]) & actual_set)
            hits.append(hit / len(actual_set) if actual_set else 0.0)
        
        recall = np.mean(hits)
        print(f"  history_days={days:<5} Recall@{k}: {recall:.6f}")

# 様々な期間で試す
days_list = [7, 14, 21, 30, 45, 60, 90, 120, 180, 365]
evaluate_history_days(transactions, customers, articles, date(2020, 9, 8), days_list)

cutoff: 2020-09-08
--------------------------------------------------
  history_days=7     Recall@12: 0.041792
  history_days=14    Recall@12: 0.045994
  history_days=21    Recall@12: 0.047773
  history_days=30    Recall@12: 0.048643
  history_days=45    Recall@12: 0.049095
  history_days=60    Recall@12: 0.048966
  history_days=90    Recall@12: 0.047996
  history_days=120   Recall@12: 0.047501
  history_days=180   Recall@12: 0.047012
  history_days=365   Recall@12: 0.045802


In [ ]:
def evaluate_history_topn(transactions, customers, articles, cutoff, topn_list, days=45, k=12):
    """historyのtop_nを変えてRecall@12を計算"""
    actuals_tmp = get_actuals(transactions, cutoff)
    global7_list = [art for art, _ in build_global_top(transactions, cutoff, 7, 50)]
    
    print(f"cutoff: {cutoff}, history_days={days}")
    print("-"*50)
    
    for top_n in topn_list:
        history = build_customer_history(transactions, cutoff, days, top_n)
        
        hits = []
        for customer_id, actual_set in actuals_tmp.items():
            history_candidates = [art for art, _, _ in history.get(customer_id, [])]
            remaining = k - len(history_candidates)
            if remaining > 0:
                history_set = set(history_candidates)
                fill = [art for art in global7_list if art not in history_set][:remaining]
                candidates = history_candidates + fill
            else:
                candidates = history_candidates[:k]
            
            hit = len(set(candidates[:k]) & actual_set)
            hits.append(hit / len(actual_set) if actual_set else 0.0)
        
        recall = np.mean(hits)
        print(f"  history_top_n={top_n:<5} Recall@{k}: {recall:.6f}")

topn_list = [6, 12, 18, 24, 30, 40, 50]
evaluate_history_topn(transactions, customers, articles, date(2020, 9, 8), topn_list)

cutoff: 2020-09-08, history_days=45
--------------------------------------------------
  history_top_n=6     Recall@12: 0.047189
  history_top_n=12    Recall@12: 0.049095
  history_top_n=18    Recall@12: 0.049095
  history_top_n=24    Recall@12: 0.049095
  history_top_n=30    Recall@12: 0.049095
  history_top_n=40    Recall@12: 0.049095
  history_top_n=50    Recall@12: 0.049095


In [ ]:
def evaluate_history_global7(transactions, customers, articles, cutoff, label_days=7, k_list=[12, 24, 48, 96, 120]):
    print(f"\ncutoff: {cutoff}")
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"valid_customers: {len(valid_customers_list):,}")
    
    # 事前計算
    history = build_customer_history(transactions, cutoff, 45, max(k_list))
    global7_list = [art for art, _ in build_global_top(transactions, cutoff, 7, max(k_list))]
    
    print("評価中...")
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for i, customer_id in enumerate(valid_customers_list):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers_list):,}")
        
        actual_set = actuals[customer_id]
        history_candidates = [art for art, _, _ in history.get(customer_id, [])]
        history_set = set(history_candidates)
        global7_fill = [art for art in global7_list if art not in history_set]
        
        for k in k_list:
            candidates = history_candidates[:k]
            remaining = k - len(candidates)
            if remaining > 0:
                candidates = candidates + global7_fill[:remaining]
            
            hit = len(set(candidates[:k]) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, candidates, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

# 評価実行
train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff = date(2020, 9, 8)

print("="*50)
print("TRAIN評価")
print("="*50)

train_results_all = []
for cutoff in train_cutoffs:
    print(f"\n--- cutoff: {cutoff} ---")
    result = evaluate_history_global7(transactions, customers, articles, cutoff)
    train_results_all.append(result)

train_avg = {}
for k in [12, 24, 48, 96, 120]:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

print("\n" + "="*50)
print("VALID評価")
print("="*50)
valid_results = evaluate_history_global7(transactions, customers, articles, valid_cutoff)

print("\n" + "="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in [12, 24, 48, 96, 120]:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in [12, 24, 48, 96, 120]:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

NameError: name 'date' is not defined

In [ ]:
from catboost import CatBoost, Pool

feature_cols = ['rank', 'item_popularity', 'days_since_last_purchase', 'is_history']

def create_reranking_dataset(transactions, cutoff, label_days=7):
    """リランキング用データセットを作成"""
    print(f"\ncutoff: {cutoff}")
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    
    # 候補生成
    history = build_customer_history(transactions, cutoff, 45, 120)
    global7_list = [art for art, _ in build_global_top(transactions, cutoff, 7, 120)]
    
    # 特徴量計算用
    # item_popularity: 直近30日の購買回数
    start_date = cutoff - timedelta(days=30)
    item_pop = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start_date)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .group_by('article_id')
        .agg(pl.len().alias('item_popularity'))
    )
    item_pop_dict = dict(zip(item_pop['article_id'].to_list(), item_pop['item_popularity'].to_list()))
    
    # user_last_purchase: ユーザーの最終購買からの経過日数
    user_last = (
        transactions
        .filter(pl.col('t_dat') <= pl.lit(cutoff))
        .group_by('customer_id')
        .agg(pl.col('t_dat').max().alias('last_date'))
    )
    user_last_dict = {
        row['customer_id']: (cutoff - row['last_date']).days
        for row in user_last.iter_rows(named=True)
    }
    
    rows = []
    for customer_id in valid_customers_list:
        actual_set = actuals[customer_id]
        history_candidates = [art for art, _, _ in history.get(customer_id, [])]
        history_set = set(history_candidates)
        global7_fill = [art for art in global7_list if art not in history_set]
        
        candidates = history_candidates[:120]
        remaining = 120 - len(candidates)
        if remaining > 0:
            candidates = candidates + global7_fill[:remaining]
        
        days_since = user_last_dict.get(customer_id, 999)
        
        for rank, article_id in enumerate(candidates, start=1):
            rows.append({
                'customer_id': customer_id,
                'article_id': article_id,
                'rank': rank,
                'item_popularity': item_pop_dict.get(article_id, 0),
                'days_since_last_purchase': days_since,
                'is_history': 1 if article_id in history_set else 0,
                'label': 1 if article_id in actual_set else 0,
            })
    
    df = pd.DataFrame(rows)
    pos = df['label'].sum()
    print(f"  行数: {len(df):,} / 正例: {pos:,} / Recall: {pos/sum(len(v) for v in actuals.values()):.4f}")
    return df

print("関数定義完了")

ModuleNotFoundError: No module named 'catboost'

In [ ]:
pip install catboost

{12: {'recall': np.float64(0.04564611826769743), 'map': np.float64(0.021308408565569936)}, 24: {'recall': np.float64(0.06267469845338403), 'map': np.float64(0.022332932319261842)}, 48: {'recall': np.float64(0.08789724653867304), 'map': np.float64(0.02318496334302392)}, 96: {'recall': np.float64(0.12468968534269775), 'map': np.float64(0.023849902437293638)}, 120: {'recall': np.float64(0.14137044083935507), 'map': np.float64(0.024056198710553825)}}


In [1]:
import polars as pl
import pandas as pd
import numpy as np
from math import log1p
from collections import defaultdict
from datetime import date, timedelta
from pathlib import Path
from catboost import CatBoost, Pool

DATA_DIR = Path('/workspace/data/raw')

HISTORY_DAYS = 45
GLOBAL_POPULARITY_DAYS = [7, 14, 30]
AGE_POPULARITY_DAYS = 14
CATEGORY_POPULARITY_DAYS = 45
CATEGORY_HISTORY_DAYS = 120
CATEGORY_TOP_N = 16
CATEGORY_PREF_TOP_N = 2
CATEGORY_COLS = ['section_name', 'garment_group_name', 'index_group_name']
EVAL_CUTOFFS = [12, 24, 48, 96, 120]
ALSO_BOUGHT_DAYS = 60
ALSO_BOUGHT_TOP_N = 20

feature_cols = ['rank', 'item_popularity', 'days_since_last_purchase', 'is_history']

print("設定完了")

設定完了


In [2]:
transactions = pl.read_csv(
    DATA_DIR / 'transactions_train.csv',
    columns=['t_dat', 'customer_id', 'article_id'],
    schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
).with_columns(pl.col('t_dat').str.to_date())

customers = pl.read_csv(
    DATA_DIR / 'customers.csv',
    columns=['customer_id', 'age'],
    schema_overrides={'customer_id': pl.String}
).with_columns(
    pl.when(pl.col('age').is_null()).then(pl.lit('unknown'))
    .when(pl.col('age') < 20).then(pl.lit('<20'))
    .when(pl.col('age') < 30).then(pl.lit('20s'))
    .when(pl.col('age') < 40).then(pl.lit('30s'))
    .when(pl.col('age') < 50).then(pl.lit('40s'))
    .when(pl.col('age') < 60).then(pl.lit('50s'))
    .otherwise(pl.lit('60+')).alias('age_bucket')
)

articles = pl.read_csv(
    DATA_DIR / 'articles.csv',
    columns=['article_id'] + CATEGORY_COLS,
    schema_overrides={'article_id': pl.String}
)

last_ts = transactions['t_dat'].max()
print(f"全データ最終日: {last_ts}, 行数: {len(transactions):,}")

全データ最終日: 2020-09-22, 行数: 31,788,324


In [3]:
def get_actuals(transactions, cutoff, label_days=7):
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

def build_global_top(transactions, cutoff, days, limit):
    start = cutoff - timedelta(days=days)
    counts = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .group_by('article_id')
        .agg(pl.len().alias('cnt'))
        .sort('cnt', descending=True)
        .head(limit)
    )
    return [(row['article_id'], row['cnt']) for row in counts.iter_rows(named=True)]

def build_customer_history(transactions, cutoff, days, top_n):
    start = cutoff - timedelta(days=days)
    recent = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .group_by(['customer_id', 'article_id'])
        .agg(pl.len().alias('cnt'), pl.max('t_dat').alias('last_date'))
        .sort(['customer_id', 'last_date', 'article_id'], descending=[False, True, False])
    )
    history = defaultdict(list)
    for row in recent.iter_rows(named=True):
        cid = row['customer_id']
        if len(history[cid]) < top_n:
            history[cid].append((row['article_id'], row['cnt'], (cutoff - row['last_date']).days))
    return dict(history)

def apk(actual_set, predicted, k):
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in seen:
            continue
        seen.add(article_id)
        if article_id in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)

print("関数定義完了")

関数定義完了


In [4]:
def create_reranking_dataset(transactions, cutoff, label_days=7):
    """リランキング用データセットを作成"""
    print(f"\ncutoff: {cutoff}")
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    
    # 候補生成
    history = build_customer_history(transactions, cutoff, 45, 120)
    global7_list = [art for art, _ in build_global_top(transactions, cutoff, 7, 120)]
    
    # item_popularity: 直近30日の購買回数
    start_date = cutoff - timedelta(days=30)
    item_pop = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start_date)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .group_by('article_id')
        .agg(pl.len().alias('item_popularity'))
    )
    item_pop_dict = dict(zip(item_pop['article_id'].to_list(), item_pop['item_popularity'].to_list()))
    
    # ユーザーの最終購買からの経過日数
    user_last = (
        transactions
        .filter(pl.col('t_dat') <= pl.lit(cutoff))
        .group_by('customer_id')
        .agg(pl.col('t_dat').max().alias('last_date'))
    )
    user_last_dict = {
        row['customer_id']: (cutoff - row['last_date']).days
        for row in user_last.iter_rows(named=True)
    }
    
    rows = []
    for customer_id in valid_customers_list:
        actual_set = actuals[customer_id]
        history_candidates = [art for art, _, _ in history.get(customer_id, [])]
        history_set = set(history_candidates)
        global7_fill = [art for art in global7_list if art not in history_set]
        
        candidates = history_candidates[:120]
        remaining = 120 - len(candidates)
        if remaining > 0:
            candidates = candidates + global7_fill[:remaining]
        
        days_since = user_last_dict.get(customer_id, 999)
        
        for rank, article_id in enumerate(candidates, start=1):
            rows.append({
                'customer_id': customer_id,
                'article_id': article_id,
                'rank': rank,
                'item_popularity': item_pop_dict.get(article_id, 0),
                'days_since_last_purchase': days_since,
                'is_history': 1 if article_id in history_set else 0,
                'label': 1 if article_id in actual_set else 0,
            })
    
    df = pd.DataFrame(rows)
    pos = df['label'].sum()
    total_actual = sum(len(v) for v in actuals.values())
    print(f"  行数: {len(df):,} / 正例: {pos:,} / Recall: {pos/total_actual:.4f}")
    return df

print("関数定義完了")

関数定義完了


In [18]:
train_cutoffs = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]

print("学習データ作成中...")
train_dfs = []
for cutoff in train_cutoffs:
    df = create_reranking_dataset(transactions, cutoff)
    train_dfs.append(df)

train_df = pd.concat(train_dfs, ignore_index=True).sort_values('customer_id')

print(f"\n学習データ合計: {len(train_df):,}行")
print(f"正例数: {train_df['label'].sum():,}")
print(f"正例率: {train_df['label'].mean():.4f}")

学習データ作成中...

cutoff: 2020-08-18
  行数: 8,644,200 / 正例: 26,142 / Recall: 0.1133

cutoff: 2020-08-25
  行数: 9,630,360 / 正例: 33,493 / Recall: 0.1313

cutoff: 2020-09-01
  行数: 9,098,640 / 正例: 32,247 / Recall: 0.1354

学習データ合計: 27,373,200行
正例数: 91,882
正例率: 0.0034


In [19]:
# CatBoost YetiRank学習
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df['label'],
    group_id=train_df['customer_id']
)

model = CatBoost({
    'loss_function': 'YetiRank',
    'iterations': 200,
    'learning_rate': 0.05,
    'depth': 6,
    'verbose': 20,
    'early_stopping_rounds': 20,
})

model.fit(train_pool)
print("学習完了")

0:	total: 10.6s	remaining: 35m 10s
20:	total: 1m 13s	remaining: 10m 27s
40:	total: 2m 17s	remaining: 8m 52s
60:	total: 3m 20s	remaining: 7m 37s
80:	total: 4m 26s	remaining: 6m 30s
100:	total: 5m 29s	remaining: 5m 23s
120:	total: 6m 33s	remaining: 4m 17s
140:	total: 7m 38s	remaining: 3m 11s
160:	total: 8m 43s	remaining: 2m 6s
180:	total: 9m 47s	remaining: 1m 1s
199:	total: 10m 48s	remaining: 0us
学習完了


In [20]:
# validデータで評価
valid_cutoff = date(2020, 9, 8)
valid_df = create_reranking_dataset(transactions, valid_cutoff)

# 予測
valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df['label'],
    group_id=valid_df['customer_id']
)

valid_df['pred_score'] = model.predict(valid_df[feature_cols])

# ユーザーごとにスコア上位K件を選択して評価
actuals_valid = get_actuals(transactions, valid_cutoff)

metric_acc = {k: {'customer_recall': [], 'map': []} for k in EVAL_CUTOFFS}

for customer_id, group in valid_df.groupby('customer_id'):
    actual_set = actuals_valid.get(customer_id, set())
    if not actual_set:
        continue
    
    prediction = group.sort_values('pred_score', ascending=False)['article_id'].tolist()
    
    for k in EVAL_CUTOFFS:
        hits = len(set(prediction[:k]) & actual_set)
        metric_acc[k]['customer_recall'].append(hits / len(actual_set))
        metric_acc[k]['map'].append(apk(actual_set, prediction, k))

print("【リランキング後の評価】")
print(f"{'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*30)
for k in EVAL_CUTOFFS:
    recall = np.mean(metric_acc[k]['customer_recall'])
    map_k = np.mean(metric_acc[k]['map'])
    print(f"{k:>4} {recall:>10.6f} {map_k:>10.6f}")


cutoff: 2020-09-08
  行数: 8,642,280 / 正例: 29,135 / Recall: 0.1278
【リランキング後の評価】
   k     recall        MAP
------------------------------
  12   0.048614   0.024255
  24   0.063707   0.025165
  48   0.086828   0.025937
  96   0.123794   0.026606
 120   0.141304   0.026824


In [21]:
# 全データで候補生成→予測→提出ファイル作成
all_customers = pl.read_csv(
    DATA_DIR / 'customers.csv',
    columns=['customer_id'],
    schema_overrides={'customer_id': pl.String}
).to_pandas()

cutoff = last_ts.date() if hasattr(last_ts, 'date') else last_ts

print("全データで候補生成中...")
history_all = build_customer_history(transactions, cutoff, 45, 120)
global7_all = [art for art, _ in build_global_top(transactions, cutoff, 7, 120)]

# item_popularity
start_date = cutoff - timedelta(days=30)
item_pop = (
    transactions
    .filter((pl.col('t_dat') > pl.lit(start_date)) & (pl.col('t_dat') <= pl.lit(cutoff)))
    .group_by('article_id')
    .agg(pl.len().alias('item_popularity'))
)
item_pop_dict = dict(zip(item_pop['article_id'].to_list(), item_pop['item_popularity'].to_list()))

# user_last_purchase
user_last = (
    transactions
    .filter(pl.col('t_dat') <= pl.lit(cutoff))
    .group_by('customer_id')
    .agg(pl.col('t_dat').max().alias('last_date'))
)
user_last_dict = {
    row['customer_id']: (cutoff - row['last_date']).days
    for row in user_last.iter_rows(named=True)
}

print("候補生成・予測中...")
predictions = {}
general_pred = ' '.join(global7_all[:12])

for customer_id in all_customers['customer_id']:
    history_candidates = [art for art, _, _ in history_all.get(customer_id, [])]
    history_set = set(history_candidates)
    global7_fill = [art for art in global7_all if art not in history_set]
    
    candidates = history_candidates[:120]
    remaining = 120 - len(candidates)
    if remaining > 0:
        candidates = candidates + global7_fill[:remaining]
    
    if not candidates:
        predictions[customer_id] = general_pred
        continue
    
    days_since = user_last_dict.get(customer_id, 999)
    
    features = pd.DataFrame([{
        'rank': rank,
        'item_popularity': item_pop_dict.get(article_id, 0),
        'days_since_last_purchase': days_since,
        'is_history': 1 if article_id in history_set else 0,
    } for rank, article_id in enumerate(candidates, start=1)])
    
    scores = model.predict(features[feature_cols])
    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    top12 = [art for art, _ in ranked[:12]]
    predictions[customer_id] = ' '.join(top12)

# 提出ファイル作成
all_customers['prediction'] = all_customers['customer_id'].map(predictions).fillna(general_pred)

OUTPUT_DIR = Path('/workspace/outputs/submissions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
all_customers.to_csv(OUTPUT_DIR / 'submission_rerank_v1.csv', index=False)

print(f"保存完了: {len(all_customers):,}ユーザー")
print(all_customers.head(5))

全データで候補生成中...
候補生成・予測中...
保存完了: 1,371,980ユーザー
                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  0568601043 0918522001 0909370001 0924243001 04...  
1  0909370001 0918522001 0448509014 0714790020 08...  
2  0794321007 0918522001 0924243001 0909370001 09...  
3  0909370001 0918522001 0448509014 0714790020 08...  
4  0896152002 0730683050 0791587015 0924243001 09...  


In [ ]:
print("全データで候補生成中...")
history_all = build_customer_history(transactions, cutoff, 45, 120)
global7_all = [art for art, _ in build_global_top(transactions, cutoff, 7, 120)]

start_date = cutoff - timedelta(days=30)
item_pop = (
    transactions
    .filter((pl.col('t_dat') > pl.lit(start_date)) & (pl.col('t_dat') <= pl.lit(cutoff)))
    .group_by('article_id')
    .agg(pl.len().alias('item_popularity'))
)
item_pop_dict = dict(zip(item_pop['article_id'].to_list(), item_pop['item_popularity'].to_list()))

user_last = (
    transactions
    .filter(pl.col('t_dat') <= pl.lit(cutoff))
    .group_by('customer_id')
    .agg(pl.col('t_dat').max().alias('last_date'))
)
user_last_dict = {
    row['customer_id']: (cutoff - row['last_date']).days
    for row in user_last.iter_rows(named=True)
}

# 全ユーザーの候補を一括で作成
print("候補データ作成中...")
rows = []
for customer_id in all_customers['customer_id']:
    history_candidates = [art for art, _, _ in history_all.get(customer_id, [])]
    history_set = set(history_candidates)
    global7_fill = [art for art in global7_all if art not in history_set]
    
    candidates = history_candidates[:120]
    remaining = 120 - len(candidates)
    if remaining > 0:
        candidates = candidates + global7_fill[:remaining]
    
    days_since = user_last_dict.get(customer_id, 999)
    
    for rank, article_id in enumerate(candidates, start=1):
        rows.append({
            'customer_id': customer_id,
            'article_id': article_id,
            'rank': rank,
            'item_popularity': item_pop_dict.get(article_id, 0),
            'days_since_last_purchase': days_since,
            'is_history': 1 if article_id in history_set else 0,
        })

all_df = pd.DataFrame(rows)
print(f"候補数: {len(all_df):,}行")

# 一括予測
print("予測中...")
all_df['pred_score'] = model.predict(all_df[feature_cols])

# ユーザーごとに上位12件を選択
print("上位12件選択中...")
submission = (
    all_df.sort_values(['customer_id', 'pred_score'], ascending=[True, False])
    .groupby('customer_id')
    .head(12)
    .groupby('customer_id')['article_id']
    .apply(lambda x: ' '.join(x))
    .reset_index()
    .rename(columns={'article_id': 'prediction'})
)

# 全ユーザーにマージ
general_pred = ' '.join(global7_all[:12])
all_customers_df = all_customers.merge(submission, on='customer_id', how='left')
all_customers_df['prediction'] = all_customers_df['prediction'].fillna(general_pred)

OUTPUT_DIR = Path('/workspace/outputs/submissions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
all_customers_df.to_csv(OUTPUT_DIR / 'submission_rerank_v1.csv', index=False)

print(f"保存完了: {len(all_customers_df):,}ユーザー")
print(all_customers_df.head(5))

全データで候補生成中...
候補データ作成中...


KeyboardInterrupt: 

In [5]:
def evaluate_timedecay_candidates(transactions, cutoff, label_days=7, k=12):
    """時間減衰スコアベースの候補生成でRecall@Kを計算"""
    a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    # ldbwを計算
    last_date = cutoff
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(last_date) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    
    # 週ごとの販売数とquotient
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == last_date)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(last_date) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    # ユーザー×商品ごとにスコアを集計
    purchase_scores = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    
    # general_pred（全体トレンド上位50件）
    general_pred = (
        df_train.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(50)
        .select('article_id').to_series().to_list()
    )
    
    # ユーザーごとにスコア上位50件を候補として抽出
    candidates_dict = {}
    for row in (
        purchase_scores
        .sort(['customer_id', 'score'], descending=[False, True])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= 50)
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in candidates_dict:
            candidates_dict[cid] = []
        candidates_dict[cid].append(row['article_id'])
    
    # Recall計算
    hits = []
    for customer_id in valid_customers_list:
        actual_set = actuals[customer_id]
        candidates = candidates_dict.get(customer_id, [])
        
        # 足りない分をgeneral_predで補完
        cand_set = set(candidates)
        fill = [art for art in general_pred if art not in cand_set]
        candidates = candidates + fill
        candidates = candidates[:k]
        
        hit = len(set(candidates) & actual_set)
        hits.append(hit / len(actual_set) if actual_set else 0.0)
    
    recall = np.mean(hits)
    print(f"  時間減衰スコア Recall@{k}: {recall:.6f}")
    return recall

# valid cutoffで計算
evaluate_timedecay_candidates(transactions, date(2020, 9, 8))

cutoff: 2020-09-08, valid_customers: 72,019
  時間減衰スコア Recall@12: 0.048041


np.float64(0.048040702155096926)

In [6]:
def evaluate_timedecay_full(transactions, customers, cutoff, label_days=7, k_list=[12, 24, 48, 96, 120]):
    """時間減衰 + general_pred + bin popularでRecall@Kを計算"""
    a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    # ldbw計算
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    
    # 週ごとの販売数とquotient
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    # ユーザー×商品ごとにスコアを集計
    purchase_scores = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    
    # general_pred（全体トレンド上位50件）
    general_pred = (
        df_train.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(50)
        .select('article_id').to_series().to_list()
    )
    
    # bin popular（年齢ビン別人気上位50件）
    customers_age = customers.with_columns(
        pl.when(pl.col('age').is_null()).then(-1)
        .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
        .alias('age_bin')
    ).select(['customer_id', 'age_bin'])
    
    df_with_age = df_train.join(customers_age, on='customer_id', how='left')
    recent = df_with_age.filter(
        pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28))
    )
    
    bin_popular_dict = {}
    for age_bin in customers_age['age_bin'].unique().to_list():
        top50 = (
            recent.filter(pl.col('age_bin') == age_bin)
            .group_by('article_id')
            .agg(pl.len().alias('count'))
            .sort('count', descending=True)
            .head(50)
            .select('article_id').to_series().to_list()
        )
        bin_popular_dict[age_bin] = top50
    
    # ユーザーごとの候補生成
    candidates_dict = {}
    for row in (
        purchase_scores
        .sort(['customer_id', 'score'], descending=[False, True])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= 50)
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in candidates_dict:
            candidates_dict[cid] = []
        candidates_dict[cid].append(row['article_id'])
    
    # age_binマップ
    age_bin_dict = dict(zip(
        customers_age['customer_id'].to_list(),
        customers_age['age_bin'].to_list()
    ))
    
    # Recall計算
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for customer_id in valid_customers_list:
        actual_set = actuals[customer_id]
        candidates = candidates_dict.get(customer_id, [])
        cand_set = set(candidates)
        
        # general_predで補完
        fill_gen = [art for art in general_pred if art not in cand_set]
        candidates = candidates + fill_gen
        cand_set = set(candidates)
        
        # bin popularで補完
        age_bin = age_bin_dict.get(customer_id, -1)
        fill_bin = [art for art in bin_popular_dict.get(age_bin, []) if art not in cand_set]
        candidates = candidates + fill_bin
        
        for k in k_list:
            top_k = candidates[:k]
            hit = len(set(top_k) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, top_k, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

# valid cutoffで計算
print("="*50)
valid_results = evaluate_timedecay_full(transactions, customers, date(2020, 9, 8))

cutoff: 2020-09-08, valid_customers: 72,019
  Recall@12: 0.047959  MAP@12: 0.026354
  Recall@24: 0.058180  MAP@24: 0.026965
  Recall@48: 0.073359  MAP@48: 0.027477
  Recall@96: 0.118501  MAP@96: 0.028230
  Recall@120: 0.127043  MAP@120: 0.028332


In [7]:
articles_full = pl.read_csv(
    DATA_DIR / 'articles.csv',
    columns=['article_id', 'product_code'],
    schema_overrides={'article_id': pl.String}
)

print(articles_full.head(5))
print(f"商品数: {len(articles_full):,}")
print(f"product_codeのユニーク数: {articles_full['product_code'].n_unique():,}")

shape: (5, 2)
┌────────────┬──────────────┐
│ article_id ┆ product_code │
│ ---        ┆ ---          │
│ str        ┆ i64          │
╞════════════╪══════════════╡
│ 0108775015 ┆ 108775       │
│ 0108775044 ┆ 108775       │
│ 0108775051 ┆ 108775       │
│ 0110065001 ┆ 110065       │
│ 0110065002 ┆ 110065       │
└────────────┴──────────────┘
商品数: 105,542
product_codeのユニーク数: 47,224


In [8]:
# product_code → article_idのマッピングを作成
product_code_map = {}
for row in articles_full.iter_rows(named=True):
    code = row['product_code']
    if code not in product_code_map:
        product_code_map[code] = []
    product_code_map[code].append(row['article_id'])

# article_id → product_codeのマッピング
article_to_product = dict(zip(
    articles_full['article_id'].to_list(),
    articles_full['product_code'].to_list()
))

print(f"product_code数: {len(product_code_map):,}")
print(f"サンプル（product_code=108775）: {product_code_map[108775]}")

product_code数: 47,224
サンプル（product_code=108775）: ['0108775015', '0108775044', '0108775051']


In [9]:
def evaluate_with_variants(transactions, customers, cutoff, label_days=7, k_list=[12, 24, 48, 96, 120]):
    """時間減衰 + general_pred + bin popular + 同一product code別バリアントでRecall@Kを計算"""
    a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    # ldbw計算
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    
    # 週ごとの販売数とquotient
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    # ユーザー×商品ごとにスコアを集計
    purchase_scores = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    
    # general_pred
    general_pred = (
        df_train.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(50)
        .select('article_id').to_series().to_list()
    )
    
    # bin popular
    customers_age = customers.with_columns(
        pl.when(pl.col('age').is_null()).then(-1)
        .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
        .alias('age_bin')
    ).select(['customer_id', 'age_bin'])
    
    df_with_age = df_train.join(customers_age, on='customer_id', how='left')
    recent = df_with_age.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
    
    bin_popular_dict = {}
    for age_bin in customers_age['age_bin'].unique().to_list():
        top50 = (
            recent.filter(pl.col('age_bin') == age_bin)
            .group_by('article_id')
            .agg(pl.len().alias('count'))
            .sort('count', descending=True)
            .head(50)
            .select('article_id').to_series().to_list()
        )
        bin_popular_dict[age_bin] = top50
    
    age_bin_dict = dict(zip(
        customers_age['customer_id'].to_list(),
        customers_age['age_bin'].to_list()
    ))
    
    # ユーザーごとの候補生成
    candidates_dict = {}
    for row in (
        purchase_scores
        .sort(['customer_id', 'score'], descending=[False, True])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= 50)
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in candidates_dict:
            candidates_dict[cid] = []
        candidates_dict[cid].append(row['article_id'])
    
    # Recall計算
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for customer_id in valid_customers_list:
        actual_set = actuals[customer_id]
        candidates = candidates_dict.get(customer_id, [])
        cand_set = set(candidates)
        
        # 同一product code別バリアントを追加
        variant_candidates = []
        for article_id in candidates:
            product_code = article_to_product.get(article_id)
            if product_code:
                variants = [a for a in product_code_map.get(product_code, [])
                           if a != article_id and a not in cand_set]
                variant_candidates.extend(variants)
                cand_set.update(variants)
        
        candidates = candidates + variant_candidates
        cand_set = set(candidates)
        
        # general_predで補完
        fill_gen = [art for art in general_pred if art not in cand_set]
        candidates = candidates + fill_gen
        cand_set = set(candidates)
        
        # bin popularで補完
        age_bin = age_bin_dict.get(customer_id, -1)
        fill_bin = [art for art in bin_popular_dict.get(age_bin, []) if art not in cand_set]
        candidates = candidates + fill_bin
        
        for k in k_list:
            top_k = candidates[:k]
            hit = len(set(top_k) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, top_k, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("="*50)
valid_results_variant = evaluate_with_variants(transactions, customers, date(2020, 9, 8))

cutoff: 2020-09-08, valid_customers: 72,019
  Recall@12: 0.049201  MAP@12: 0.026483
  Recall@24: 0.059929  MAP@24: 0.027188
  Recall@48: 0.077026  MAP@48: 0.027803
  Recall@96: 0.108450  MAP@96: 0.028383
  Recall@120: 0.118670  MAP@120: 0.028501


In [10]:
def evaluate_single_strategies(transactions, customers, cutoff, k=12):
    """各戦略の単体Recall@kを計算"""
    a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
    
    actuals = get_actuals(transactions, cutoff)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    # 共通データ準備
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    # 時間減衰スコア
    purchase_scores = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    timedecay_dict = {}
    for row in (
        purchase_scores
        .sort(['customer_id', 'score'], descending=[False, True])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= 50)
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in timedecay_dict:
            timedecay_dict[cid] = []
        timedecay_dict[cid].append(row['article_id'])
    
    # general_pred
    general_pred = (
        df_train.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(50)
        .select('article_id').to_series().to_list()
    )
    
    # bin popular
    customers_age = customers.with_columns(
        pl.when(pl.col('age').is_null()).then(-1)
        .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
        .alias('age_bin')
    ).select(['customer_id', 'age_bin'])
    
    df_with_age = df_train.join(customers_age, on='customer_id', how='left')
    recent = df_with_age.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
    
    bin_popular_dict = {}
    for age_bin in customers_age['age_bin'].unique().to_list():
        top50 = (
            recent.filter(pl.col('age_bin') == age_bin)
            .group_by('article_id')
            .agg(pl.len().alias('count'))
            .sort('count', descending=True)
            .head(50)
            .select('article_id').to_series().to_list()
        )
        bin_popular_dict[age_bin] = top50
    
    age_bin_dict = dict(zip(
        customers_age['customer_id'].to_list(),
        customers_age['age_bin'].to_list()
    ))
    
    # 各戦略の単体Recall計算
    strategies = {
        'timedecay': lambda cid: timedecay_dict.get(cid, []),
        'general_pred': lambda cid: general_pred,
        'bin_popular': lambda cid: bin_popular_dict.get(age_bin_dict.get(cid, -1), []),
        'variant': lambda cid: list(dict.fromkeys([
            v for art in timedecay_dict.get(cid, [])
            for v in product_code_map.get(article_to_product.get(art), [])
            if v != art
        ])),
    }
    
    print("\n【各戦略の単体Recall@12】")
    print("-"*40)
    results = {}
    for name, get_cands in strategies.items():
        hits = []
        for customer_id in valid_customers_list:
            actual_set = actuals[customer_id]
            candidates = get_cands(customer_id)[:k]
            hit = len(set(candidates) & actual_set)
            hits.append(hit / len(actual_set) if actual_set else 0.0)
        recall = np.mean(hits)
        results[name] = recall
        print(f"  {name:<20} Recall@{k}: {recall:.6f}")
    
    print("\n【優先順位（高い順）】")
    for name, recall in sorted(results.items(), key=lambda x: -x[1]):
        print(f"  {name:<20} {recall:.6f}")
    
    return results

evaluate_single_strategies(transactions, customers, date(2020, 9, 8))

cutoff: 2020-09-08, valid_customers: 72,019

【各戦略の単体Recall@12】
----------------------------------------
  timedecay            Recall@12: 0.043646
  general_pred         Recall@12: 0.013786
  bin_popular          Recall@12: 0.024077
  variant              Recall@12: 0.018639

【優先順位（高い順）】
  timedecay            0.043646
  bin_popular          0.024077
  variant              0.018639
  general_pred         0.013786


{'timedecay': np.float64(0.04364617093809777),
 'general_pred': np.float64(0.013785519911562844),
 'bin_popular': np.float64(0.024076727648697817),
 'variant': np.float64(0.018639198965122367)}

In [11]:
def evaluate_single_strategies_with_fallback(transactions, customers, cutoff, k=12):
    """各戦略の単体Recall@k（直近1週間人気商品で補完）"""
    a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
    
    actuals = get_actuals(transactions, cutoff)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    # 共通データ準備
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    # 共通補完用：直近1週間の人気商品
    global7_fallback = [art for art, _ in build_global_top(transactions, cutoff, 7, k)]
    
    # 時間減衰スコア
    purchase_scores = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    timedecay_dict = {}
    for row in (
        purchase_scores
        .sort(['customer_id', 'score'], descending=[False, True])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= k)
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in timedecay_dict:
            timedecay_dict[cid] = []
        timedecay_dict[cid].append(row['article_id'])
    
    # general_pred
    general_pred = (
        df_train.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(k)
        .select('article_id').to_series().to_list()
    )
    
    # bin popular
    customers_age = customers.with_columns(
        pl.when(pl.col('age').is_null()).then(-1)
        .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
        .alias('age_bin')
    ).select(['customer_id', 'age_bin'])
    
    df_with_age = df_train.join(customers_age, on='customer_id', how='left')
    recent = df_with_age.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
    
    bin_popular_dict = {}
    for age_bin in customers_age['age_bin'].unique().to_list():
        top_k = (
            recent.filter(pl.col('age_bin') == age_bin)
            .group_by('article_id')
            .agg(pl.len().alias('count'))
            .sort('count', descending=True)
            .head(k)
            .select('article_id').to_series().to_list()
        )
        bin_popular_dict[age_bin] = top_k
    
    age_bin_dict = dict(zip(
        customers_age['customer_id'].to_list(),
        customers_age['age_bin'].to_list()
    ))
    
    # variant
    def get_variants(cid):
        variants = []
        seen = set()
        for art in timedecay_dict.get(cid, []):
            product_code = article_to_product.get(art)
            if product_code:
                for v in product_code_map.get(product_code, []):
                    if v != art and v not in seen:
                        variants.append(v)
                        seen.add(v)
        return variants
    
    # 各戦略の定義
    strategies = {
        'timedecay'   : lambda cid: timedecay_dict.get(cid, []),
        'general_pred': lambda cid: general_pred,
        'bin_popular' : lambda cid: bin_popular_dict.get(age_bin_dict.get(cid, -1), []),
        'variant'     : lambda cid: get_variants(cid),
    }
    
    def fill_with_fallback(candidates, k):
        """直近1週間の人気商品で補完"""
        cand_set = set(candidates)
        fill = [art for art in global7_fallback if art not in cand_set]
        return (candidates + fill)[:k]
    
    print("\n【各戦略の単体Recall@12（global7で補完）】")
    print("-"*40)
    results = {}
    for name, get_cands in strategies.items():
        hits = []
        for customer_id in valid_customers_list:
            actual_set = actuals[customer_id]
            candidates = fill_with_fallback(get_cands(customer_id), k)
            hit = len(set(candidates) & actual_set)
            hits.append(hit / len(actual_set) if actual_set else 0.0)
        recall = np.mean(hits)
        results[name] = recall
        print(f"  {name:<20} Recall@{k}: {recall:.6f}")
    
    print("\n【優先順位（高い順）】")
    for name, recall in sorted(results.items(), key=lambda x: -x[1]):
        print(f"  {name:<20} {recall:.6f}")
    
    return results

evaluate_single_strategies_with_fallback(transactions, customers, date(2020, 9, 8))

cutoff: 2020-09-08, valid_customers: 72,019

【各戦略の単体Recall@12（global7で補完）】
----------------------------------------
  timedecay            Recall@12: 0.050002
  general_pred         Recall@12: 0.013786
  bin_popular          Recall@12: 0.024105
  variant              Recall@12: 0.021721

【優先順位（高い順）】
  timedecay            0.050002
  bin_popular          0.024105
  variant              0.021721
  general_pred         0.013786


{'timedecay': np.float64(0.050002212718208656),
 'general_pred': np.float64(0.013785519911562846),
 'bin_popular': np.float64(0.024104696458515856),
 'variant': np.float64(0.02172092874244625)}

In [12]:
def evaluate_two_strategies(transactions, customers, cutoff, label_days=7, k_list=[12, 24, 48, 96, 120]):
    """2つの候補生成戦略を比較"""
    a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    # 共通データ準備
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    # 時間減衰スコア
    purchase_scores = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    timedecay_dict = {}
    for row in (
        purchase_scores
        .sort(['customer_id', 'score'], descending=[False, True])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= 120)
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in timedecay_dict:
            timedecay_dict[cid] = []
        timedecay_dict[cid].append(row['article_id'])
    
    # global7（補完用）
    global7 = [art for art, _ in build_global_top(transactions, cutoff, 7, 120)]
    
    # general_pred
    general_pred = (
        df_train.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(120)
        .select('article_id').to_series().to_list()
    )
    
    # bin popular
    customers_age = customers.with_columns(
        pl.when(pl.col('age').is_null()).then(-1)
        .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
        .alias('age_bin')
    ).select(['customer_id', 'age_bin'])
    
    df_with_age = df_train.join(customers_age, on='customer_id', how='left')
    recent = df_with_age.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
    
    bin_popular_dict = {}
    for age_bin in customers_age['age_bin'].unique().to_list():
        top = (
            recent.filter(pl.col('age_bin') == age_bin)
            .group_by('article_id')
            .agg(pl.len().alias('count'))
            .sort('count', descending=True)
            .head(120)
            .select('article_id').to_series().to_list()
        )
        bin_popular_dict[age_bin] = top
    
    age_bin_dict = dict(zip(
        customers_age['customer_id'].to_list(),
        customers_age['age_bin'].to_list()
    ))
    
    def get_variants(cid, cand_set):
        variants = []
        for art in timedecay_dict.get(cid, []):
            product_code = article_to_product.get(art)
            if product_code:
                for v in product_code_map.get(product_code, []):
                    if v != art and v not in cand_set:
                        variants.append(v)
                        cand_set.add(v)
        return variants
    
    def fill_candidates(candidates, fills, k):
        cand_set = set(candidates)
        for fill_list in fills:
            for art in fill_list:
                if art not in cand_set:
                    candidates.append(art)
                    cand_set.add(art)
        return candidates[:k]
    
    # 2戦略の評価
    strategy_names = [
        'timedecay + global7',
        'timedecay → bin_popular → variant → general_pred'
    ]
    all_results = {}
    
    for strategy_name in strategy_names:
        print(f"\n--- {strategy_name} ---")
        metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
        
        for customer_id in valid_customers_list:
            actual_set = actuals[customer_id]
            td_cands = list(timedecay_dict.get(customer_id, []))
            cand_set = set(td_cands)
            
            if strategy_name == 'timedecay + global7':
                fills = [global7]
            else:
                age_bin = age_bin_dict.get(customer_id, -1)
                bin_pop = bin_popular_dict.get(age_bin, [])
                variants = get_variants(customer_id, cand_set)
                fills = [bin_pop, variants, general_pred]
            
            for k in k_list:
                candidates = fill_candidates(list(td_cands), fills, k)
                hit = len(set(candidates[:k]) & actual_set)
                metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
                metric_acc[k]['map'].append(apk(actual_set, candidates[:k], k))
        
        results = {}
        for k in k_list:
            results[k] = {
                'recall': np.mean(metric_acc[k]['customer_recall']),
                'map': np.mean(metric_acc[k]['map'])
            }
            print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
        
        all_results[strategy_name] = results
    
    # 比較表
    print("\n" + "="*60)
    print("比較結果")
    print("="*60)
    print(f"{'戦略':<45} {'k':>4} {'recall':>10} {'MAP':>10}")
    print("-"*75)
    for name, results in all_results.items():
        for k in k_list:
            print(f"{name:<45} {k:>4} {results[k]['recall']:>10.6f} {results[k]['map']:>10.6f}")
        print("-"*75)
    
    return all_results

evaluate_two_strategies(transactions, customers, date(2020, 9, 8))

cutoff: 2020-09-08, valid_customers: 72,019

--- timedecay + global7 ---
  Recall@12: 0.050020  MAP@12: 0.026573
  Recall@24: 0.062673  MAP@24: 0.027324
  Recall@48: 0.086825  MAP@48: 0.028100
  Recall@96: 0.124522  MAP@96: 0.028766
  Recall@120: 0.141884  MAP@120: 0.028980

--- timedecay → bin_popular → variant → general_pred ---
  Recall@12: 0.051054  MAP@12: 0.026779
  Recall@24: 0.065797  MAP@24: 0.027661
  Recall@48: 0.089918  MAP@48: 0.028431
  Recall@96: 0.127810  MAP@96: 0.029106
  Recall@120: 0.142131  MAP@120: 0.029279

比較結果
戦略                                               k     recall        MAP
---------------------------------------------------------------------------
timedecay + global7                             12   0.050020   0.026573
timedecay + global7                             24   0.062673   0.027324
timedecay + global7                             48   0.086825   0.028100
timedecay + global7                             96   0.124522   0.028766
timedecay + global

{'timedecay + global7': {12: {'recall': np.float64(0.05002031680367244),
   'map': np.float64(0.026572723961656625)},
  24: {'recall': np.float64(0.06267345189433969),
   'map': np.float64(0.027323678814368924)},
  48: {'recall': np.float64(0.0868250601252346),
   'map': np.float64(0.028099997167154687)},
  96: {'recall': np.float64(0.12452152226115873),
   'map': np.float64(0.028766008845048117)},
  120: {'recall': np.float64(0.14188364040568982),
   'map': np.float64(0.028979641736039934)}},
 'timedecay → bin_popular → variant → general_pred': {12: {'recall': np.float64(0.05105375323531208),
   'map': np.float64(0.02677905208694898)},
  24: {'recall': np.float64(0.06579694253107171),
   'map': np.float64(0.02766072643343436)},
  48: {'recall': np.float64(0.08991762303084881),
   'map': np.float64(0.028430820652166407)},
  96: {'recall': np.float64(0.1278099775710343),
   'map': np.float64(0.029106008101667757)},
  120: {'recall': np.float64(0.1421307790065344),
   'map': np.float64(0

In [13]:
import polars as pl
import pandas as pd
import numpy as np
from math import log1p
from collections import defaultdict
from datetime import date, timedelta
from pathlib import Path
from catboost import CatBoost, Pool

DATA_DIR = Path('/workspace/data/raw')
OUTPUT_DIR = Path('/workspace/outputs/submissions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
CANDIDATE_LIMIT = 24
EVAL_CUTOFFS = [12, 24, 48, 96, 120]

feature_cols = ['rank', 'score', 'item_popularity', 'days_since_last_purchase', 'is_history', 'is_variant']

print("設定完了")

設定完了


In [14]:
transactions = pl.read_csv(
    DATA_DIR / 'transactions_train.csv',
    columns=['t_dat', 'customer_id', 'article_id'],
    schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
).with_columns(pl.col('t_dat').str.to_date())

customers = pl.read_csv(
    DATA_DIR / 'customers.csv',
    columns=['customer_id', 'age'],
    schema_overrides={'customer_id': pl.String}
).with_columns(
    pl.when(pl.col('age').is_null()).then(-1)
    .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
    .alias('age_bin')
)

articles_full = pl.read_csv(
    DATA_DIR / 'articles.csv',
    columns=['article_id', 'product_code'],
    schema_overrides={'article_id': pl.String}
)

# product_codeマッピング
product_code_map = {}
article_to_product = {}
for row in articles_full.iter_rows(named=True):
    code = row['product_code']
    art = row['article_id']
    article_to_product[art] = code
    if code not in product_code_map:
        product_code_map[code] = []
    product_code_map[code].append(art)

last_ts = transactions['t_dat'].max()
print(f"全データ最終日: {last_ts}, 行数: {len(transactions):,}")

全データ最終日: 2020-09-22, 行数: 31,788,324


In [15]:
def get_actuals(transactions, cutoff, label_days=7):
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

def build_global_top(transactions, cutoff, days, limit):
    start = cutoff - timedelta(days=days)
    counts = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .group_by('article_id')
        .agg(pl.len().alias('cnt'))
        .sort('cnt', descending=True)
        .head(limit)
    )
    return [(row['article_id'], row['cnt']) for row in counts.iter_rows(named=True)]

def apk(actual_set, predicted, k):
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in seen:
            continue
        seen.add(article_id)
        if article_id in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)

def build_timedecay_candidates(transactions, cutoff, top_n=120):
    """時間減衰スコアベースの候補生成"""
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    purchase_scores = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    
    general_pred = (
        df_train.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(top_n)
        .select('article_id').to_series().to_list()
    )
    
    timedecay_dict = {}
    score_dict = {}
    for row in (
        purchase_scores
        .sort(['customer_id', 'score'], descending=[False, True])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in timedecay_dict:
            timedecay_dict[cid] = []
            score_dict[cid] = {}
        timedecay_dict[cid].append(row['article_id'])
        score_dict[cid][row['article_id']] = row['score']
    
    return timedecay_dict, score_dict, general_pred, df_train

def build_bin_popular(df_train, customers, cutoff, top_n=120):
    """年齢ビン別人気商品"""
    age_bin_dict = dict(zip(
        customers['customer_id'].to_list(),
        customers['age_bin'].to_list()
    ))
    
    df_with_age = df_train.join(
        customers.select(['customer_id', 'age_bin']),
        on='customer_id', how='left'
    )
    recent = df_with_age.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
    
    bin_popular_dict = {}
    for age_bin in customers['age_bin'].unique().to_list():
        top = (
            recent.filter(pl.col('age_bin') == age_bin)
            .group_by('article_id')
            .agg(pl.len().alias('count'))
            .sort('count', descending=True)
            .head(top_n)
            .select('article_id').to_series().to_list()
        )
        bin_popular_dict[age_bin] = top
    
    return bin_popular_dict, age_bin_dict

def generate_candidates(customer_id, timedecay_dict, score_dict, bin_popular_dict, 
                        age_bin_dict, general_pred, n_candidates=24):
    """候補生成: timedecay → bin_popular → variant → general_pred"""
    td_cands = list(timedecay_dict.get(customer_id, []))
    cand_set = set(td_cands)
    
    # bin popularで補完
    age_bin = age_bin_dict.get(customer_id, -1)
    bin_pop = [art for art in bin_popular_dict.get(age_bin, []) if art not in cand_set]
    
    # variantで補完
    variants = []
    for art in td_cands:
        product_code = article_to_product.get(art)
        if product_code:
            for v in product_code_map.get(product_code, []):
                if v != art and v not in cand_set:
                    variants.append(v)
                    cand_set.add(v)
    
    # general_predで補完
    gen = [art for art in general_pred if art not in cand_set]
    
    candidates = td_cands + bin_pop + variants + gen
    return candidates[:n_candidates]

print("関数定義完了")

関数定義完了


In [16]:
def create_dataset(transactions, customers, cutoff, label_days=7, n_candidates=24):
    """リランキング用データセット作成"""
    print(f"\ncutoff: {cutoff}")
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    
    timedecay_dict, score_dict, general_pred, df_train = build_timedecay_candidates(transactions, cutoff)
    bin_popular_dict, age_bin_dict = build_bin_popular(df_train, customers, cutoff)
    
    # item_popularity
    start_date = cutoff - timedelta(days=30)
    item_pop = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start_date)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .group_by('article_id')
        .agg(pl.len().alias('item_popularity'))
    )
    item_pop_dict = dict(zip(item_pop['article_id'].to_list(), item_pop['item_popularity'].to_list()))
    
    # user_last_purchase
    user_last = (
        transactions
        .filter(pl.col('t_dat') <= pl.lit(cutoff))
        .group_by('customer_id')
        .agg(pl.col('t_dat').max().alias('last_date'))
    )
    user_last_dict = {
        row['customer_id']: (cutoff - row['last_date']).days
        for row in user_last.iter_rows(named=True)
    }
    
    rows = []
    for customer_id in valid_customers_list:
        actual_set = actuals[customer_id]
        td_cands = set(timedecay_dict.get(customer_id, []))
        
        # variant候補のset
        variant_set = set()
        for art in timedecay_dict.get(customer_id, []):
            product_code = article_to_product.get(art)
            if product_code:
                for v in product_code_map.get(product_code, []):
                    if v != art:
                        variant_set.add(v)
        
        candidates = generate_candidates(
            customer_id, timedecay_dict, score_dict,
            bin_popular_dict, age_bin_dict, general_pred, n_candidates
        )
        
        days_since = user_last_dict.get(customer_id, 999)
        
        for rank, article_id in enumerate(candidates, start=1):
            rows.append({
                'customer_id': customer_id,
                'article_id': article_id,
                'rank': rank,
                'score': score_dict.get(customer_id, {}).get(article_id, 0.0),
                'item_popularity': item_pop_dict.get(article_id, 0),
                'days_since_last_purchase': days_since,
                'is_history': 1 if article_id in td_cands else 0,
                'is_variant': 1 if article_id in variant_set else 0,
                'label': 1 if article_id in actual_set else 0,
            })
    
    df = pd.DataFrame(rows)
    pos = df['label'].sum()
    total_actual = sum(len(v) for v in actuals.values())
    print(f"  行数: {len(df):,} / 正例: {pos:,} / Recall: {pos/total_actual:.4f}")
    return df

print("関数定義完了")

関数定義完了


In [17]:
train_cutoffs = [
    date(2020, 9, 15),
    date(2020, 9, 8),
    date(2020, 9, 1),
    date(2020, 8, 25),
    date(2020, 8, 18),
]

print("学習データ作成中...")
models = []

for fold, cutoff in enumerate(train_cutoffs, 1):
    print(f"\n{'='*40}")
    print(f"Fold {fold}/5")
    print(f"{'='*40}")
    
    df = create_dataset(transactions, customers, cutoff)
    df = df.sort_values('customer_id')
    
    train_pool = Pool(
        data=df[feature_cols],
        label=df['label'],
        group_id=df['customer_id']
    )
    
    model = CatBoost({
        'loss_function': 'YetiRank',
        'iterations': 200,
        'learning_rate': 0.05,
        'depth': 6,
        'verbose': 20,
        'early_stopping_rounds': 20,
    })
    model.fit(train_pool)
    models.append(model)
    print(f"Fold {fold} 完了")

print("\n全fold学習完了")

学習データ作成中...

Fold 1/5

cutoff: 2020-09-15
  行数: 1,655,616 / 正例: 10,928 / Recall: 0.0511
0:	total: 299ms	remaining: 59.5s
20:	total: 3.74s	remaining: 31.9s
40:	total: 7.23s	remaining: 28s
60:	total: 10.6s	remaining: 24.1s
80:	total: 14s	remaining: 20.5s
100:	total: 17.4s	remaining: 17s
120:	total: 20.7s	remaining: 13.5s
140:	total: 24.3s	remaining: 10.2s
160:	total: 28.1s	remaining: 6.81s
180:	total: 31.5s	remaining: 3.31s
199:	total: 34.7s	remaining: 0us
Fold 1 完了

Fold 2/5

cutoff: 2020-09-08
  行数: 1,728,456 / 正例: 11,479 / Recall: 0.0504
0:	total: 269ms	remaining: 53.6s
20:	total: 4.13s	remaining: 35.2s
40:	total: 7.78s	remaining: 30.2s
60:	total: 11.3s	remaining: 25.8s
80:	total: 15s	remaining: 22s
100:	total: 18.5s	remaining: 18.1s
120:	total: 22.1s	remaining: 14.4s
140:	total: 25.8s	remaining: 10.8s
160:	total: 29.7s	remaining: 7.19s
180:	total: 33.5s	remaining: 3.52s
199:	total: 37.2s	remaining: 0us
Fold 2 完了

Fold 3/5

cutoff: 2020-09-01
  行数: 1,819,728 / 正例: 11,801 / Recall: 0.0

In [18]:
valid_cutoff = date(2020, 9, 8)
valid_df = create_dataset(transactions, customers, valid_cutoff)
valid_df['pred_score'] = np.mean([m.predict(valid_df[feature_cols]) for m in models], axis=0)

actuals_valid = get_actuals(transactions, valid_cutoff)
metric_acc = {k: {'customer_recall': [], 'map': []} for k in EVAL_CUTOFFS}

for customer_id, group in valid_df.groupby('customer_id'):
    actual_set = actuals_valid.get(customer_id, set())
    if not actual_set:
        continue
    prediction = group.sort_values('pred_score', ascending=False)['article_id'].tolist()
    for k in EVAL_CUTOFFS:
        hits = len(set(prediction[:k]) & actual_set)
        metric_acc[k]['customer_recall'].append(hits / len(actual_set))
        metric_acc[k]['map'].append(apk(actual_set, prediction, k))

print("【valid評価】")
print(f"{'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*30)
for k in EVAL_CUTOFFS:
    recall = np.mean(metric_acc[k]['customer_recall'])
    map_k = np.mean(metric_acc[k]['map'])
    print(f"{k:>4} {recall:>10.6f} {map_k:>10.6f}")


cutoff: 2020-09-08
  行数: 1,728,456 / 正例: 11,468 / Recall: 0.0503
【valid評価】
   k     recall        MAP
------------------------------
  12   0.055472   0.029163
  24   0.065743   0.029806
  48   0.065743   0.029803
  96   0.065743   0.029803
 120   0.065743   0.029803


In [19]:
from tqdm import tqdm

print("提出ファイル作成中...")
cutoff = last_ts

# 全データで候補生成
print("候補生成中...")
timedecay_dict_all, score_dict_all, general_pred_all, df_train_all = build_timedecay_candidates(transactions, cutoff)
bin_popular_dict_all, age_bin_dict_all = build_bin_popular(df_train_all, customers, cutoff)

# item_popularity
start_date = cutoff - timedelta(days=30)
item_pop = (
    transactions
    .filter((pl.col('t_dat') > pl.lit(start_date)) & (pl.col('t_dat') <= pl.lit(cutoff)))
    .group_by('article_id')
    .agg(pl.len().alias('item_popularity'))
)
item_pop_dict = dict(zip(item_pop['article_id'].to_list(), item_pop['item_popularity'].to_list()))

# user_last_purchase
user_last = (
    transactions
    .filter(pl.col('t_dat') <= pl.lit(cutoff))
    .group_by('customer_id')
    .agg(pl.col('t_dat').max().alias('last_date'))
)
user_last_dict = {
    row['customer_id']: (cutoff - row['last_date']).days
    for row in user_last.iter_rows(named=True)
}

# 全ユーザーの候補を一括作成
all_customers = pl.read_csv(
    DATA_DIR / 'customers.csv',
    columns=['customer_id'],
    schema_overrides={'customer_id': pl.String}
)['customer_id'].to_list()

general_pred_top12 = ' '.join(general_pred_all[:12])

print("候補データ作成中...")
rows = []
for customer_id in tqdm(all_customers, desc="候補生成"):
    td_cands = set(timedecay_dict_all.get(customer_id, []))
    
    variant_set = set()
    for art in timedecay_dict_all.get(customer_id, []):
        product_code = article_to_product.get(art)
        if product_code:
            for v in product_code_map.get(product_code, []):
                if v != art:
                    variant_set.add(v)
    
    candidates = generate_candidates(
        customer_id, timedecay_dict_all, score_dict_all,
        bin_popular_dict_all, age_bin_dict_all, general_pred_all, CANDIDATE_LIMIT
    )
    
    if not candidates:
        continue
    
    days_since = user_last_dict.get(customer_id, 999)
    
    for rank, article_id in enumerate(candidates, start=1):
        rows.append({
            'customer_id': customer_id,
            'article_id': article_id,
            'rank': rank,
            'score': score_dict_all.get(customer_id, {}).get(article_id, 0.0),
            'item_popularity': item_pop_dict.get(article_id, 0),
            'days_since_last_purchase': days_since,
            'is_history': 1 if article_id in td_cands else 0,
            'is_variant': 1 if article_id in variant_set else 0,
        })

all_df = pd.DataFrame(rows)
print(f"候補数: {len(all_df):,}行")

# 一括予測（5foldアンサンブル）
print("予測中...")
all_df['pred_score'] = np.mean([m.predict(all_df[feature_cols]) for m in models], axis=0)

# ユーザーごとに上位12件を選択
print("上位12件選択中...")
submission = (
    all_df.sort_values(['customer_id', 'pred_score'], ascending=[True, False])
    .groupby('customer_id')
    .head(12)
    .groupby('customer_id')['article_id']
    .apply(lambda x: ' '.join(x))
    .reset_index()
    .rename(columns={'article_id': 'prediction'})
)

# 全ユーザーにマージ
all_customers_df = pd.DataFrame({'customer_id': all_customers})
all_customers_df = all_customers_df.merge(submission, on='customer_id', how='left')
all_customers_df['prediction'] = all_customers_df['prediction'].fillna(general_pred_top12)

# 保存
all_customers_df.to_csv(OUTPUT_DIR / 'submission_rerank_v2.csv', index=False)
print(f"保存完了: {OUTPUT_DIR / 'submission_rerank_v2.csv'}")
print(f"ユーザー数: {len(all_customers_df):,}")
print(all_customers_df.head(5))

提出ファイル作成中...
候補生成中...
候補データ作成中...


候補生成: 100%|██████████| 1371980/1371980 [01:39<00:00, 13849.20it/s]


候補数: 32,927,520行
予測中...
上位12件選択中...
保存完了: /workspace/outputs/submissions/submission_rerank_v2.csv
ユーザー数: 1,371,980
                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  0568601043 0568601006 0863595006 0751471001 07...  
1  0826211002 0739590027 0759871002 0764280001 05...  
2  0794321007 0852643001 0909370001 0852643003 04...  
3  0896152002 0751471001 0751471038 0751471043 07...  
4  0896152002 0730683050 0791587015 0863595006 07...  


In [20]:
train_cutoffs_eval = [
    date(2020, 8, 18),
    date(2020, 8, 25),
    date(2020, 9, 1),
]
valid_cutoff_eval = date(2020, 9, 8)

def evaluate_candidate_generation(transactions, customers, cutoff, label_days=7, k_list=[12, 24, 48, 96, 120]):
    """候補生成のRecall@KとMAP@Kを計算"""
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    timedecay_dict, score_dict, general_pred, df_train = build_timedecay_candidates(transactions, cutoff)
    bin_popular_dict, age_bin_dict = build_bin_popular(df_train, customers, cutoff)
    
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for customer_id in valid_customers_list:
        actual_set = actuals[customer_id]
        candidates = generate_candidates(
            customer_id, timedecay_dict, score_dict,
            bin_popular_dict, age_bin_dict, general_pred, max(k_list)
        )
        for k in k_list:
            top_k = candidates[:k]
            hit = len(set(top_k) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, top_k, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
    return results

# train評価
print("="*50)
print("TRAIN評価")
print("="*50)
train_results_all = []
for cutoff in train_cutoffs_eval:
    print(f"\n--- cutoff: {cutoff} ---")
    result = evaluate_candidate_generation(transactions, customers, cutoff)
    train_results_all.append(result)

train_avg = {}
for k in [12, 24, 48, 96, 120]:
    train_avg[k] = {
        'recall': np.mean([r[k]['recall'] for r in train_results_all]),
        'map': np.mean([r[k]['map'] for r in train_results_all])
    }

# valid評価
print("\n" + "="*50)
print("VALID評価")
print("="*50)
valid_results = evaluate_candidate_generation(transactions, customers, valid_cutoff_eval)

# 結果表示
print("\n" + "="*60)
print(f"{'split':<12} {'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*40)
for k in [12, 24, 48, 96, 120]:
    print(f"{'train avg':<12} {k:>4} {train_avg[k]['recall']:>10.6f} {train_avg[k]['map']:>10.6f}")
print("-"*40)
for k in [12, 24, 48, 96, 120]:
    print(f"{'valid':<12} {k:>4} {valid_results[k]['recall']:>10.6f} {valid_results[k]['map']:>10.6f}")

TRAIN評価

--- cutoff: 2020-08-18 ---
cutoff: 2020-08-18, valid_customers: 72,035

--- cutoff: 2020-08-25 ---
cutoff: 2020-08-25, valid_customers: 80,253

--- cutoff: 2020-09-01 ---
cutoff: 2020-09-01, valid_customers: 75,822

VALID評価
cutoff: 2020-09-08, valid_customers: 72,019

split           k     recall        MAP
----------------------------------------
train avg      12   0.045730   0.023251
train avg      24   0.058158   0.024000
train avg      48   0.077398   0.024630
train avg      96   0.107999   0.025161
train avg     120   0.120519   0.025307
----------------------------------------
valid          12   0.051061   0.026702
valid          24   0.065787   0.027583
valid          48   0.089959   0.028356
valid          96   0.127732   0.029029
valid         120   0.142222   0.029204
